# S6E9: T4 x2

T4 x2で候補探索、RealMLP比較、seed平均、cross-fitted blend、最終5-foldを実行します。出力は `/kaggle/working/submission.csv` です。


In [ ]:
from pathlib import Path
import importlib.util
import sys
import subprocess

required = ['numpy', 'pandas', 'sklearn', 'xgboost', 'lightgbm', 'catboost', 'optuna', 'joblib', 'scipy']
missing = [name for name in required if importlib.util.find_spec(name) is None]
assert not missing, f'Missing libraries: {missing}'

PYTABKIT_WHEEL = ('https://files.pythonhosted.org/packages/0d/48/'
    '75c0c38ea2b086cd967b7241f394773437a1a7c09dc9b178b49f04399207/'
    'pytabkit-1.7.3-py3-none-any.whl')
install = None
if importlib.util.find_spec('pytabkit') is None:
    install = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-deps', '-q', PYTABKIT_WHEEL],
        text=True, capture_output=True)
probe = subprocess.run(
    [sys.executable, '-c', 'from pytabkit import RealMLP_TD_Classifier'],
    text=True, capture_output=True)
PYTABKIT_AVAILABLE = probe.returncode == 0
if not PYTABKIT_AVAILABLE:
    detail = (probe.stderr or (install.stderr if install else '') or 'installation unavailable')[-2000:]
    print('RealMLP disabled; PyTabKit could not be loaded:\n' + detail)
print('Dependencies ready. RealMLP:', PYTABKIT_AVAILABLE)


In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import shutil


def _is_s6e9_data(folder):
    required = ('train.csv', 'test.csv', 'sample_submission.csv')
    try:
        headers = []
        for filename in required:
            with (folder / filename).open(encoding='utf-8-sig', newline='') as stream:
                headers.append(next(csv.reader(stream)))
        train, test, sample = headers
        if any(len(h) != len(set(h)) for h in headers):
            return False
        target = 'Will_Buy_EV'
        return (set(sample) == {'id', target}
                and {'id', target, 'Age', 'Annual_Income_USD', 'Daily_Commute_km'} <= set(train)
                and target not in test and set(train) - {target} == set(test))
    except (OSError, UnicodeError, StopIteration, csv.Error):
        return False


def find_s6e9_data(input_root='/kaggle/input', override=None):
    root = Path(input_root)
    if override is not None:
        folder = Path(override)
        if _is_s6e9_data(folder):
            return folder
        raise FileNotFoundError(
            f'DATA_OVERRIDE={folder} に、S6E9の train.csv / test.csv / sample_submission.csv '
            'が揃っているか、CSVの列名を確認してください。')
    for folder in (root / 'competitions/playground-series-s6e9', root / 'playground-series-s6e9'):
        if _is_s6e9_data(folder):
            return folder
    matches = sorted({p.parent for p in root.rglob('train.csv') if _is_s6e9_data(p.parent)}) if root.is_dir() else []
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError('対応するデータが複数あります。DATA_OVERRIDE に使うフォルダーを指定してください:\n'
                         + '\n'.join(str(p) for p in matches))
    available = '\n'.join(f'  - {p.name}' for p in sorted(root.iterdir())[:20]) if root.is_dir() else '  （入力なし）'
    raise FileNotFoundError(
        'S6E9の学習データが見つかりません。\n'
        '右側の Add Input からコンペ本体「Predicting Electric Vehicle Purchases」'
        '（playground-series-s6e9）を追加し、このセルを再実行してください。\n'
        '必要なファイル: train.csv / test.csv / sample_submission.csv\n'
        '0.946xx.csv などの提出予測CSVだけのデータセットでは学習できません。\n'
        'ZIPをUploadした場合は、CSVを展開したフォルダーを DATA_OVERRIDE に指定してください。\n'
        f'現在の入力:\n{available}')


def restore_previous_run(run, input_root='/kaggle/input', override=None):
    run, root = Path(run), Path(input_root)
    if run.exists():
        return None
    if override is not None:
        candidates = [Path(override)]
    else:
        candidates = sorted({p.parent for p in root.rglob('frozen.json')
                             if p.parent.name == run.name and (p.parent / 'config.json').is_file()}) if root.is_dir() else []
    valid = [folder for folder in candidates
             if (folder / 'frozen.json').is_file() and (folder / 'config.json').is_file()]
    if len(valid) > 1:
        raise ValueError('Multiple resumable runs found; set RESUME_RUN to one exact folder:\n'
                         + '\n'.join(str(folder) for folder in valid))
    if not valid:
        if override is not None:
            raise FileNotFoundError(f'RESUME_RUN is not a frozen S6E9 run: {override}')
        return None
    shutil.copytree(valid[0], run)
    print('Restored previous run:', valid[0])
    return valid[0]


def _artifact_digest(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def _write_artifact_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def migrate_restored_run(run, gpu_ids, source_root='/kaggle/working'):
    run, source_root = Path(run), Path(source_root)
    config_path, frozen_path = run / 'config.json', run / 'frozen.json'
    if not config_path.is_file() or not frozen_path.is_file():
        return False
    config = json.loads(config_path.read_text(encoding='utf-8'))
    current = {name: _artifact_digest(source_root / name) for name in config.get('sources', {})}
    changed = sorted(name for name, old_digest in config.get('sources', {}).items()
                     if current.get(name) != old_digest)
    gpu_ids = list(map(str, gpu_ids))
    gpu_changed = config.get('gpu_ids') != gpu_ids
    if not changed and not gpu_changed:
        return False
    if changed not in ([], ['s6e9/audit.py']) or (changed and (run / 'sealed_report.json').exists()):
        raise ValueError(f'Unsafe resume: changed sources are {changed}; use a fresh RUN.')
    frozen = json.loads(frozen_path.read_text(encoding='utf-8'))
    if frozen.get('config_sha256') != _artifact_digest(config_path):
        raise ValueError('Cannot migrate a changed frozen configuration.')
    marker_path = run / 'SEALED_OPENED.json'
    marker = json.loads(marker_path.read_text(encoding='utf-8')) if marker_path.exists() else None
    if marker is not None and marker.get('frozen_sha256') != _artifact_digest(frozen_path):
        raise ValueError('Cannot migrate a changed frozen selection.')
    previous_config = _artifact_digest(config_path)
    if changed:
        config['sources']['s6e9/audit.py'] = current['s6e9/audit.py']
    config['gpu_ids'] = gpu_ids
    _write_artifact_json(config_path, config)
    frozen['config_sha256'] = _artifact_digest(config_path)
    _write_artifact_json(frozen_path, frozen)
    if marker is not None:
        marker['frozen_sha256'] = _artifact_digest(frozen_path)
        _write_artifact_json(marker_path, marker)
    _write_artifact_json(run / 'resume_migration.json', {
        'reason': 'audit numpy import and/or ephemeral GPU selectors', 'previous_config_sha256': previous_config,
        'config_sha256': _artifact_digest(config_path), 'frozen_sha256': _artifact_digest(frozen_path)})
    print('Migrated restored run for the audit fix and current GPU selectors.')
    return True


DATA_OVERRIDE = None
DATA = find_s6e9_data(override=DATA_OVERRIDE)
RUN = Path('/kaggle/working/s6e9_diversity_v6_0')
RESUME_RUN = None  # Exact prior output folder, or None for unique auto-detection
restore_previous_run(RUN, override=RESUME_RUN)
GPU_IDS = None  # T4 x2を自動検出。手動指定なら['0', '1']
DOMAIN_COMPARE = True  # GPU互換設定でドメイン特徴を比較
CAT_COMPARE = True  # 固定CatBoostをT4で比較
REALMLP_COMPARE = PYTABKIT_AVAILABLE
FINAL_SEEDS = [2026, 42, 3407]
LGB_TRIALS = 12  # 完了候補の総数。freeze前なら40などに増やして続行可能
XGB_TRIALS = 6   # 同上。追加探索なら20など
SCREEN_FOLDS = 2
PROMOTE_TRIALS = 3
SEED = 2026
MAX_ROUNDS = 3500
EARLY_STOPPING = 120
THREADS = 4  # 並列プロセス全体のCPUスレッド予算
BUILD_CUDA_IF_NEEDED = True
print('Data:', DATA, 'Outputs:', RUN)


## セットアップ
packageを展開し、T4 x2を確認します。


In [ ]:
PACKAGE = Path('/kaggle/working/s6e9')
PACKAGE.mkdir(parents=True, exist_ok=True)
(PACKAGE / '__init__.py').write_text('"""S6E9 experiment pipeline."""\n\nfrom .config import LGB_CUDA_BIN_LIMIT, SEED, TARGET\n\n__all__ = ["TARGET", "SEED", "LGB_CUDA_BIN_LIMIT"]\n', encoding='utf-8')
(PACKAGE / 'artifact.py').write_text('"""Artifact serialization and integrity helpers."""\nimport hashlib\nimport json\n\nimport numpy as np\nfrom scipy.stats import rankdata\nfrom pathlib import Path\n\n\ndef write_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for block in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\n\ndef key(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()\n\n\ndef checked_prediction(path):\n    meta = read_json(path.with_suffix(\'.json\'))\n    if digest(path) != meta[\'sha256\']:\n        raise ValueError(f\'Corrupt prediction cache: {path}\')\n    pred = np.load(path, allow_pickle=False)\n    if not np.isfinite(pred).all() or not ((pred >= 0) & (pred <= 1)).all():\n        raise ValueError(f\'Invalid probabilities: {path}\')\n    return pred, meta\n\n\ndef write_runtime_profile(run):\n    groups = {}\n    for path in (Path(run) / \'cache\' / \'predictions\').glob(\'*.json\'):\n        record = read_json(path)\n        if \'prepare_seconds\' not in record or \'fit_predict_seconds\' not in record:\n            continue\n        name = f"{record.get(\'device\', \'unknown\')}:{record.get(\'family\', \'unknown\')}:{record.get(\'phase\', \'unknown\')}"\n        group = groups.setdefault(name, dict(jobs=0, feature_cache_hits=0,\n            prepare_seconds=0., fit_predict_seconds=0.))\n        group[\'jobs\'] += 1\n        group[\'feature_cache_hits\'] += int(record.get(\'feature_cache_hit\', False))\n        group[\'prepare_seconds\'] += float(record[\'prepare_seconds\'])\n        group[\'fit_predict_seconds\'] += float(record[\'fit_predict_seconds\'])\n    for group in groups.values():\n        group[\'prepare_seconds\'] = round(group[\'prepare_seconds\'], 3)\n        group[\'fit_predict_seconds\'] = round(group[\'fit_predict_seconds\'], 3)\n    write_json(Path(run) / \'runtime_profile.json\', dict(\n        note=\'Summed worker time; parallel jobs overlap in wall-clock time.\', groups=groups))\n\n\ndef write_screen_diagnostics(run):\n    """Measure whether two-fold screening preserves the promoted order."""\n    manifest_path = run / \'promoted.json\'\n    if not manifest_path.exists():\n        return\n    manifest = read_json(manifest_path)\n    lanes = {}\n    for lane, names in manifest.get(\'lanes\', {}).items():\n        records = []\n        for name in names:\n            path = run / \'candidates\' / f\'{name}.json\'\n            if not path.exists():\n                continue\n            record = read_json(path)\n            if record.get(\'screen_auc\') is None or record.get(\'dev_auc\') is None:\n                continue\n            records.append(record)\n        if not records:\n            continue\n        screen_values = np.array([record[\'screen_auc\'] for record in records], dtype=float)\n        full_values = np.array([record[\'dev_auc\'] for record in records], dtype=float)\n        screen_ranks = rankdata(-screen_values, method=\'average\')\n        full_ranks = rankdata(-full_values, method=\'average\')\n        correlation = None\n        if len(records) > 1 and np.std(screen_ranks) > 0 and np.std(full_ranks) > 0:\n            correlation = float(np.corrcoef(screen_ranks, full_ranks)[0, 1])\n        entries = []\n        for record, screen_rank, full_rank in zip(records, screen_ranks, full_ranks):\n            entries.append(dict(name=record[\'name\'], screen_auc=record[\'screen_auc\'],\n                full_auc=record[\'dev_auc\'], delta_auc=record[\'dev_auc\'] - record[\'screen_auc\'],\n                screen_rank=float(screen_rank), full_rank=float(full_rank),\n                screen_fold_auc=record.get(\'screen_fold_auc\'),\n                full_fold_auc=record.get(\'full_fold_auc\')))\n        lanes[lane] = dict(promoted_count=len(records), rank_correlation=correlation,\n                           candidates=entries)\n    write_json(run / \'screen_promotion_diagnostics.json\', dict(\n        screen_folds=manifest.get(\'screen_folds\'),\n        note=\'Diagnostic only; screening folds and promotion counts are unchanged.\', lanes=lanes))\n', encoding='utf-8')
(PACKAGE / 'audit.py').write_text('"""Sealed holdout audit after selection is frozen."""\nimport numpy as np\nfrom sklearn.metrics import roc_auc_score\n\nimport prototype as p\nfrom .artifact import write_runtime_profile\nfrom .freeze import blend_values, frozen_config, lexicographic_rank\nfrom .tuning import context, folds\n\n\ndef audit(args):\n    run, cfg, tr, te, sub, id_col, cols, y, (dev, sealed, cv, outer) = context(args)\n    frozen = frozen_config(run)\n    if (run / \'sealed_report.json\').exists():\n        print(\'Existing holdout report retained.\', flush=True)\n        return\n    p.write_json(run / \'SEALED_OPENED.json\', dict(frozen_sha256=p.digest(run / \'frozen.json\')))\n    values = {c[\'name\']: folds(args, cfg, c, \'final\', [4])[4][0][:len(sealed)]\n              for c in frozen[\'candidates\']}\n    pred = blend_values(np.column_stack([values[c[\'name\']] for c in frozen[\'candidates\']]),\n                        frozen[\'weights\'], frozen[\'mode\'])\n    tie = frozen.get(\'tie_breaker\')\n    if tie is not None:\n        candidate = tie[\'candidate\']\n        secondary = values.get(candidate[\'name\'])\n        if secondary is None:\n            secondary = folds(args, cfg, candidate, \'final\', [4])[4][0][:len(sealed)]\n        pred = lexicographic_rank(pred, secondary)\n    baseline = folds(args, cfg, frozen[\'baseline\'], \'final\', [4])[4][0][:len(sealed)]\n    p.write_json(run / \'sealed_report.json\', dict(sealed_auc=float(roc_auc_score(y[sealed], pred)),\n        baseline_auc=float(roc_auc_score(y[sealed], baseline)), public_lb=None,\n        note=\'Previously reviewed holdout split: reference only, not a fresh independent evaluation. Do not retune.\'))\n    write_runtime_profile(run)\n    print(p.read_json(run / \'sealed_report.json\'), flush=True)\n', encoding='utf-8')
(PACKAGE / 'config.py').write_text('"""Shared immutable defaults."""\n\nTARGET = "Will_Buy_EV"\nSEED = 2026\nLGB_CUDA_BIN_LIMIT = 255\n', encoding='utf-8')
(PACKAGE / 'data.py').write_text('"""Competition data loading and schema validation."""\nfrom pathlib import Path\nimport pandas as pd\n\nfrom .config import TARGET\n\n\ndef load_data(data):\n    data = Path(data)\n    required = ["train.csv", "test.csv", "sample_submission.csv"]\n    if not all((data / f).is_file() for f in required):\n        raise FileNotFoundError(f"Place {required} in {data}; no artificial fallback is used.")\n    tr, te, sub = [pd.read_csv(data / f) for f in required]\n    if TARGET not in tr or TARGET in te or TARGET not in sub:\n        raise ValueError(f"Expected binary target {TARGET}")\n    ids = [c for c in sub if c != TARGET]\n    if len(ids) != 1:\n        raise ValueError("Expected one submission ID column")\n    id_col = ids[0]\n    if set(tr[TARGET].unique()) == {"No", "Yes"}:\n        tr[TARGET] = tr[TARGET].map({"No": 0, "Yes": 1})\n    if not set(tr[TARGET].unique()) == {0, 1}:\n        raise ValueError("Target must contain exactly 0 and 1, with no missing labels")\n    for frame in (tr, te, sub):\n        if id_col not in frame or frame[id_col].isna().any() or frame[id_col].duplicated().any():\n            raise ValueError("Missing or duplicate IDs")\n    if set(te[id_col]) != set(sub[id_col]) or len(te) != len(sub):\n        raise ValueError("Test/submission IDs differ")\n    if set(tr[id_col]) & set(te[id_col]):\n        raise ValueError("Train/test IDs overlap")\n    features = [c for c in tr if c not in (id_col, TARGET)]\n    if set(features) != set(te.columns) - {id_col}:\n        raise ValueError("Train/test feature schemas differ")\n    return tr, te, sub, id_col, features\n', encoding='utf-8')
(PACKAGE / 'features.py').write_text('"""Feature construction fitted only on each training partition."""\nimport hashlib\nimport re\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import KFold\n\nfrom .config import SEED\n\n\ndef augment(x, variant):\n    x = x.copy().reset_index(drop=True)\n    # Stable internal names work with all three libraries, including LightGBM.\n    original = list(x)\n    x.columns = [f"f{i}" for i in range(len(original))]\n    if variant in ("interaction", "artifact"):\n        numeric = [c for c in x if pd.api.types.is_numeric_dtype(x[c])]\n        for c in numeric:\n            values = x[c].astype(float)\n            if variant == "interaction":\n                x[c + "_log"] = np.sign(values) * np.log1p(np.abs(values))\n            else:\n                x[c + "_fraction"] = values - np.floor(values)\n                x[c + "_rounded"] = values.round(1)\n        lookup = {re.sub(r"[^a-z0-9]", "", c.lower()): f"f{i}" for i, c in enumerate(original)}\n        pairs = [("Annual_Income", "Vehicle_Cost"), ("Monthly_Income", "Vehicle_Cost"),\n                 ("Daily_Commute_Distance", "Charging_Stations_Nearby"),\n                 ("Daily_Usage_km", "Battery_Range_km")]\n        for a, b in pairs:\n            ca, cb = [lookup.get(re.sub(r"[^a-z0-9]", "", c.lower())) for c in (a, b)]\n            if ca in numeric and cb in numeric:\n                x[f"{ca}_over_{cb}"] = x[ca] / (x[cb].abs() + 1)\n        cats = [c for c in x if not pd.api.types.is_numeric_dtype(x[c])]\n        # A bounded generic interaction set; variant must earn its place in dev CV.\n        for a, b in zip(cats[:4], cats[1:5]):\n            x[f"{a}_cross_{b}"] = x[a].astype("string").fillna("<NA>") + "|" + x[b].astype("string").fillna("<NA>")\n        # S6E9 domain features: deterministic, no target or validation statistics.\n        names = {c: f"f{i}" for i, c in enumerate(original)}\n        home, work = names.get("Charging_Stations_Near_Home"), names.get("Charging_Stations_Near_Work")\n        commute = names.get("Daily_Commute_km")\n        if home is not None and work is not None:\n            x["stations_total"] = x[home] + x[work]\n            x["stations_gap"] = x[home] - x[work]\n            if commute is not None:\n                x["commute_per_station"] = x[commute] / (1 + x["stations_total"])\n        anxiety = names.get("Range_Anxiety_Level")\n        if anxiety is not None:\n            x["anxiety_ordinal"] = x[anxiety].map({"Low": 0, "Medium": 1, "High": 2})\n        income, cars = names.get("Annual_Income_USD"), names.get("Number_of_Cars_Owned")\n        if income is not None and cars is not None:\n            x["income_per_car"] = x[income] / (1 + x[cars])\n    return x\n\n\nSIGNAL_VARIANTS = ("numeric_te", "digits_te", "multiscale_te", "multiscale_dual")\n\n\nclass Features:\n    """All learned mappings fit on this fit partition only, including frequency/TE."""\n    def __init__(self, variant="raw", seed=SEED):\n        self.variant, self.seed = variant, seed\n\n    @staticmethod\n    def tokens(s):\n        return s.astype("string").fillna("<MISSING>")\n\n    def te_map(self, s, y):\n        stats = pd.DataFrame({"key": s.to_numpy(), "y": np.asarray(y)}).groupby("key")["y"].agg(["sum", "count"])\n        prior = float(np.mean(y))\n        return (stats["sum"] + 20 * prior) / (stats["count"] + 20), prior\n\n    def fit_transform(self, x, y):\n        z = augment(x, self.variant)\n        self.cats = [c for c in z if not pd.api.types.is_numeric_dtype(z[c])]\n        self.maps = {c: {v: i + 1 for i, v in enumerate(sorted(self.tokens(z[c]).unique()))} for c in self.cats}\n        self.freq_cols = list(z) if self.variant in ("frequency", "artifact") else []\n        self.freq = {c: self.tokens(z[c]).value_counts(normalize=True) for c in self.freq_cols}\n        self.te = {c: self.te_map(self.tokens(z[c]), y) for c in self.cats} if self.variant == "target" else {}\n        result = self._transform_augmented(z, include_te=False)\n        # KFold assignment does not depend on labels. Both map AND prior exclude each row.\n        if self.te:\n            splits = list(KFold(4, shuffle=True, random_state=self.seed).split(z))\n            for c in self.cats:\n                tokens = self.tokens(z[c])\n                values = np.empty(len(z), dtype=np.float32)\n                for tr, va in splits:\n                    mapping, prior = self.te_map(tokens.iloc[tr], np.asarray(y)[tr])\n                    values[va] = tokens.iloc[va].map(mapping).fillna(prior)\n                result[c + "_te"] = values\n        return result\n\n    def transform(self, x):\n        z = augment(x, self.variant)\n        return self._transform_augmented(z)\n\n    def _transform_augmented(self, z, include_te=True):\n        out = z.copy()\n        for c in self.cats:\n            codes = self.tokens(z[c]).map(self.maps[c]).fillna(0).astype(int)\n            out[c] = pd.Categorical(codes, categories=range(len(self.maps[c]) + 1))\n        for c in self.freq_cols:\n            out[c + "_freq"] = self.tokens(z[c]).map(self.freq[c]).fillna(0).astype(np.float32)\n        for c, (mapping, prior) in (self.te.items() if include_te else []):\n            out[c + "_te"] = self.tokens(z[c]).map(mapping).fillna(prior).astype(np.float32)\n        for c in out:\n            if c not in self.cats:\n                out[c] = pd.to_numeric(out[c], errors="coerce").replace([np.inf, -np.inf], np.nan).astype(np.float32)\n        return out\n\n\nclass SignalFeatures:\n    """Numeric/digit keys with fit-only frequency and label-independent cross-fit TE.\n\n    Inspired by Naji\'s S6E9 feature experiments; implementation is independent.\n    https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94606-lb-0-94637\n    No reference submission, test labels, or external labels are used.\n    """\n    def __init__(self, variant, seed=SEED):\n        self.variant, self.seed = variant, seed\n        self.smoothing = (10., 100.) if variant == "multiscale_dual" else (20.,)\n\n    def keys(self, x):\n        # Build keys before float32 conversion: income digits can be lost on downcast.\n        x = x.reset_index(drop=True)\n        keys = {f"k{i}": Features.tokens(x[c]) for i, c in enumerate(x)}\n        numeric = {}\n        if self.variant != "numeric_te":\n            for i, c in enumerate(x):\n                if not pd.api.types.is_numeric_dtype(x[c]) or c == "Number_of_Cars_Owned":\n                    continue\n                values = pd.to_numeric(x[c], errors="coerce").to_numpy(dtype=np.float64)\n                for power in range(-4, 4):\n                    name = f"d{i}_{power + 4}"\n                    with np.errstate(invalid="ignore"):\n                        digit = np.floor_divide(values, 10. ** power) % 10\n                    numeric[name] = digit.astype(np.float32)\n                    keys[name] = Features.tokens(pd.Series(digit))\n        if self.variant in ("multiscale_te", "multiscale_dual"):\n            scales = {"Annual_Income_USD": (1., 100., 1000.), "Daily_Commute_km": (1., 5., 10.)}\n            for column, widths in scales.items():\n                if column not in x:\n                    continue\n                values = pd.to_numeric(x[column], errors="coerce").to_numpy(dtype=np.float64)\n                for j, width in enumerate(widths):\n                    name = f"b{list(x).index(column)}_{j}"\n                    bins = np.floor(values / width)\n                    numeric[name] = bins.astype(np.float32)\n                    keys[name] = Features.tokens(pd.Series(bins))\n        return pd.DataFrame(keys), pd.DataFrame(numeric, index=x.index)\n\n    def fit_transform(self, x, y):\n        y = np.asarray(y, dtype=np.float64)\n        self.base = Features("raw", self.seed)\n        base = self.base.fit_transform(x, y)\n        keys, numeric = self.keys(x)\n        # Fit-only pruning: validation/test values do not select columns.\n        self.keep = []\n        seen = set()\n        for c in keys:\n            if keys[c].nunique(dropna=False) <= 1:\n                continue\n            signature = hashlib.sha256(pd.util.hash_pandas_object(keys[c], index=False).values.tobytes()).hexdigest()\n            if signature not in seen:\n                seen.add(signature)\n                self.keep.append(c)\n        self.numeric_keep = [c for c in numeric if c in self.keep]\n        self.maps = {}\n        splits = list(KFold(4, shuffle=True, random_state=self.seed).split(y))\n        extras = {c: numeric[c].to_numpy() for c in self.numeric_keep}\n        for c in self.keep:\n            codes, levels = pd.factorize(keys[c], sort=True)\n            size = len(levels)\n            counts = np.bincount(codes, minlength=size).astype(float)\n            sums = np.bincount(codes, weights=y, minlength=size)\n            prior = float(y.mean())\n            frequency = counts / len(y)\n            full = [(sums + strength * prior) / (counts + strength) for strength in self.smoothing]\n            self.maps[c] = (pd.Index(levels), frequency, full, prior)\n            extras[c + "_freq"] = frequency[codes].astype(np.float32)\n            oof = np.empty((len(y), len(self.smoothing)), dtype=np.float32)\n            for it, iv in splits:\n                inner_count = np.bincount(codes[it], minlength=size)\n                inner_sum = np.bincount(codes[it], weights=y[it], minlength=size)\n                inner_prior = float(y[it].mean())\n                for j, strength in enumerate(self.smoothing):\n                    mapping = (inner_sum + strength * inner_prior) / (inner_count + strength)\n                    oof[iv, j] = mapping[codes[iv]]\n            for j, strength in enumerate(self.smoothing):\n                extras[f"{c}_te{int(strength)}"] = oof[:, j]\n        return pd.concat([base, pd.DataFrame(extras, index=base.index)], axis=1)\n\n    def transform(self, x):\n        base = self.base.transform(x)\n        keys, numeric = self.keys(x)\n        extras = {c: numeric[c].to_numpy() for c in self.numeric_keep}\n        for c in self.keep:\n            levels, frequency, full, prior = self.maps[c]\n            codes = levels.get_indexer(keys[c])\n            known = codes >= 0\n            freq = np.zeros(len(x), dtype=np.float32)\n            freq[known] = frequency[codes[known]]\n            extras[c + "_freq"] = freq\n            for strength, mapping in zip(self.smoothing, full):\n                values = np.full(len(x), prior, dtype=np.float32)\n                values[known] = mapping[codes[known]]\n                extras[f"{c}_te{int(strength)}"] = values\n        return pd.concat([base, pd.DataFrame(extras, index=base.index)], axis=1)\n\n\ndef domain_features(x):\n    """Deterministic EV interactions; no labels or fitted statistics."""\n    x = x.reset_index(drop=True)\n    values = {}\n    def num(name):\n        return pd.to_numeric(x[name], errors=\'coerce\').astype(np.float64)\n    home, work = \'Charging_Stations_Near_Home\', \'Charging_Stations_Near_Work\'\n    if home in x and work in x:\n        total = num(home) + num(work)\n        values[\'domain_stations_total\'] = total\n        values[\'domain_stations_gap\'] = num(home) - num(work)\n        if \'Daily_Commute_km\' in x:\n            values[\'domain_commute_per_station\'] = num(\'Daily_Commute_km\') / (1 + total)\n    if \'Annual_Income_USD\' in x and \'Number_of_Cars_Owned\' in x:\n        values[\'domain_income_per_car\'] = num(\'Annual_Income_USD\') / (1 + num(\'Number_of_Cars_Owned\'))\n    if \'Range_Anxiety_Level\' in x:\n        values[\'domain_anxiety_ordinal\'] = x[\'Range_Anxiety_Level\'].map({\'Low\': 0, \'Medium\': 1, \'High\': 2})\n    return pd.DataFrame(values, index=x.index).replace([np.inf, -np.inf], np.nan)\n\n\nclass DomainSignalFeatures(SignalFeatures):\n    """Keep every multiscale_dual feature unchanged; add domain values, frequency and TE."""\n    def __init__(self, seed=SEED):\n        super().__init__(\'multiscale_dual\', seed)\n\n    def keys(self, x):\n        keys, numeric = super().keys(x)\n        for name, values in domain_features(x).items():\n            keys[name] = Features.tokens(values)\n            numeric[name] = values.astype(np.float32)\n        return keys, numeric\n\n\ndef make_features(variant, seed=SEED):\n    if variant == \'multiscale_domain\':\n        return DomainSignalFeatures(seed)\n    return SignalFeatures(variant, seed) if variant in SIGNAL_VARIANTS else Features(variant, seed)\n', encoding='utf-8')
(PACKAGE / 'finalize.py').write_text('"""Final five-fold OOF and submission artifact generation."""\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\n\nimport prototype as p\nfrom .artifact import write_runtime_profile\nfrom .freeze import blend_values, frozen_config, lexicographic_rank\nfrom .tuning import context, folds\n\n\ndef finalize(args):\n    run, cfg, tr, te, sub, id_col, cols, y, (dev, sealed, cv, outer) = context(args)\n    frozen = frozen_config(run)\n    if not (run / \'sealed_report.json\').exists():\n        raise ValueError(\'Run audit after freeze first.\')\n    candidates = list(frozen[\'candidates\'])\n    tie = frozen.get(\'tie_breaker\')\n    if tie is not None and tie[\'candidate\'][\'name\'] not in {c[\'name\'] for c in candidates}:\n        candidates.append(tie[\'candidate\'])\n    predictions = {c[\'name\']: folds(args, cfg, c, \'final\', list(range(5))) for c in candidates}\n    oof, test_folds, test_secondary_folds = np.empty(len(y)), [], []\n    oof_secondary = np.empty(len(y)) if tie is not None else None\n    for fold in range(5):\n        iv = np.flatnonzero(outer == fold)\n        vm = np.column_stack([predictions[c[\'name\']][fold][0][:len(iv)] for c in frozen[\'candidates\']])\n        tm = np.column_stack([predictions[c[\'name\']][fold][0][len(iv):] for c in frozen[\'candidates\']])\n        valid_blend = blend_values(vm, frozen[\'weights\'], frozen[\'mode\'])\n        test_blend = blend_values(tm, frozen[\'weights\'], frozen[\'mode\'])\n        if tie is not None:\n            secondary = predictions[tie[\'candidate\'][\'name\']][fold][0]\n            oof_secondary[iv] = secondary[:len(iv)]\n            test_secondary_folds.append(secondary[len(iv):])\n        oof[iv] = valid_blend\n        test_folds.append(test_blend)\n    test = np.mean(test_folds, axis=0)\n    if tie is not None:\n        oof = lexicographic_rank(oof, oof_secondary)\n        test = lexicographic_rank(test, np.mean(test_secondary_folds, axis=0))\n    sub[p.TARGET] = sub[id_col].map(pd.Series(test, index=te[id_col]))\n    if sub[p.TARGET].isna().any() or not sub[p.TARGET].between(0, 1).all():\n        raise ValueError(\'Invalid submission probabilities or ID alignment\')\n    sub.to_csv(run / \'submission.csv\', index=False)\n    result = pd.DataFrame({id_col: tr[id_col], p.TARGET: y, \'fold\': outer, \'prediction\': oof})\n    for c in candidates:\n        model_predictions = predictions[c[\'name\']]\n        raw_oof = np.empty(len(y))\n        for fold in range(5):\n            iv = np.flatnonzero(outer == fold)\n            raw_oof[iv] = model_predictions[fold][0][:len(iv)]\n        result[c[\'name\']] = raw_oof\n    result.to_csv(run / \'final_oof.csv\', index=False)\n    p.write_json(run / \'final_report.json\', dict(final_oof_auc=float(roc_auc_score(y, oof)),\n        public_lb=None, submission_sha256=p.digest(run / \'submission.csv\'),\n        note=\'OOF includes candidate/weight selection bias. Public LB remains unmeasured.\'))\n    write_runtime_profile(run)\n    print(f\'Created {run / "submission.csv"}\', flush=True)\n', encoding='utf-8')
(PACKAGE / 'freeze.py').write_text('"""Development-only candidate, seed, blend, and tie-break selection."""\nimport itertools\nfrom pathlib import Path\n\nimport numpy as np\nimport optuna\nfrom scipy.stats import rankdata\nfrom sklearn.metrics import roc_auc_score\n\nimport prototype as p\nfrom .tuning import context, evaluate\n\n\ndef load_candidate(run, name, y, dev, sealed):\n    path = run / \'candidates\' / name\n    record = p.read_json(path.with_suffix(\'.json\'))\n    if p.digest(path.with_suffix(\'.npy\')) != record[\'oof_sha256\']:\n        raise ValueError(f\'Changed OOF: {name}\')\n    oof = np.load(path.with_suffix(\'.npy\'), allow_pickle=False)\n    if oof.shape != y.shape or not np.isnan(oof[sealed]).all() or not np.isfinite(oof[dev]).all():\n        raise ValueError(f\'Invalid OOF or holdout contamination: {name}\')\n    if not np.isclose(roc_auc_score(y[dev], oof[dev]), record[\'dev_auc\'], atol=1e-12, rtol=0):\n        raise ValueError(f\'OOF score mismatch: {name}\')\n    return record, oof\n\n\ndef weight_grid(size, max_secondary=.10):\n    if not 0 <= max_secondary <= .30:\n        raise ValueError(\'Secondary blend cap must be between 0 and 0.30\')\n    unit = np.zeros(size)\n    unit[0] = 1\n    yield unit.copy()\n    fractions = [fraction for fraction in (.01, .02, .03, .05, .07, .10, .15, .20, .25, .30)\n                 if fraction <= max_secondary + 1e-12]\n    for i in range(1, size):\n        for fraction in fractions:\n            w = unit.copy()\n            w[0], w[i] = 1 - fraction, fraction\n            yield w\n    for i, j in itertools.combinations(range(1, size), 2):\n        for a in fractions:\n            for b in fractions:\n                if a + b > max_secondary + 1e-12:\n                    continue\n                w = unit.copy()\n                w[0], w[i], w[j] = 1 - a - b, a, b\n                yield w\n\n\ndef rank_columns(matrix):\n    ranked = np.empty(matrix.shape, dtype=float)\n    for column in range(matrix.shape[1]):\n        ranked[:, column] = (rankdata(matrix[:, column], method=\'average\') - .5) / len(matrix)\n    return ranked\n\n\ndef rank_oof_matrix(matrix, cv):\n    ranked = np.full(matrix.shape, np.nan, dtype=float)\n    for _, iv in cv:\n        ranked[iv] = rank_columns(matrix[iv])\n    return ranked\n\n\ndef blend_values(matrix, weights, mode):\n    if mode not in (\'probability\', \'rank\'):\n        raise ValueError(f\'Unknown blend mode: {mode}\')\n    values = rank_columns(matrix) if mode == \'rank\' else matrix\n    return values @ np.asarray(weights)\n\n\ndef fold_auc_scores(y, prediction, cv):\n    return [float(roc_auc_score(y[iv], prediction[iv])) for _, iv in cv]\n\n\ndef best_blend(y, values, rows, max_secondary):\n    weights = next(weight_grid(values.shape[1], max_secondary))\n    score = float(roc_auc_score(y[rows], values[rows] @ weights))\n    for candidate in weight_grid(values.shape[1], max_secondary):\n        candidate_score = float(roc_auc_score(y[rows], values[rows] @ candidate))\n        if candidate_score > score + 1e-6:\n            score, weights = candidate_score, candidate.copy()\n    return score, weights\n\n\ndef crossfit_blend(y, values, cv, max_secondary):\n    prediction = np.full(len(y), np.nan)\n    weights = []\n    for fold, (_, iv) in enumerate(cv):\n        training = np.concatenate([other_iv for other_fold, (_, other_iv) in enumerate(cv)\n                                   if other_fold != fold])\n        _, selected = best_blend(y, values, training, max_secondary)\n        prediction[iv] = values[iv] @ selected\n        weights.append(selected.tolist())\n    development = np.concatenate([iv for _, iv in cv])\n    return dict(auc=float(roc_auc_score(y[development], prediction[development])),\n                fold_auc=fold_auc_scores(y, prediction, cv), fold_weights=weights,\n                prediction=prediction)\n\n\ndef improvement_gate(base, candidate):\n    wins = sum(right > left + 1e-6 for left, right in zip(base[\'fold_auc\'], candidate[\'fold_auc\']))\n    return candidate[\'auc\'] > base[\'auc\'] + 1e-6 and wins >= 3, wins\n\n\ndef rank_blend_allowed(probability_auc, rank_auc, probability_folds, rank_folds):\n    wins = sum(rank > probability + 1e-6\n               for probability, rank in zip(probability_folds, rank_folds))\n    return rank_auc > probability_auc + 1e-6 and wins >= 3, wins\n\n\ndef tie_break_allowed(base_auc, tie_auc, base_folds, tie_folds):\n    wins = sum(tie > base for base, tie in zip(base_folds, tie_folds))\n    return tie_auc > base_auc and wins >= 3, wins\n\n\ndef lexicographic_rank(primary, secondary):\n    primary, secondary = np.asarray(primary), np.asarray(secondary)\n    if primary.ndim != 1 or primary.shape != secondary.shape or not len(primary):\n        raise ValueError(\'Tie-breaking inputs must be non-empty aligned vectors\')\n    if not np.isfinite(primary).all() or not np.isfinite(secondary).all():\n        raise ValueError(\'Tie-breaking inputs must be finite\')\n    order = np.lexsort((secondary, primary))\n    ordered_primary, ordered_secondary = primary[order], secondary[order]\n    starts = np.r_[0, 1 + np.flatnonzero(\n        (ordered_primary[1:] != ordered_primary[:-1]) |\n        (ordered_secondary[1:] != ordered_secondary[:-1]))]\n    ends = np.r_[starts[1:], len(primary)]\n    average_ranks = (starts + 1 + ends) / 2\n    ranked = np.empty(len(primary), dtype=float)\n    ranked[order] = np.repeat(average_ranks, ends - starts)\n    return (ranked - .5) / len(primary)\n\n\ndef lexicographic_oof(primary, secondary, cv):\n    result = np.full(len(primary), np.nan, dtype=float)\n    development = np.concatenate([iv for _, iv in cv])\n    result[development] = lexicographic_rank(primary[development], secondary[development])\n    return result\n\n\ndef fold_tie_statistics(primary, cv):\n    development = np.concatenate([iv for _, iv in cv])\n    counts = np.unique(primary[development], return_counts=True)[1]\n    tied = counts[counts > 1]\n    per_fold = []\n    for fold, (_, iv) in enumerate(cv):\n        counts = np.unique(primary[iv], return_counts=True)[1]\n        fold_ties = counts[counts > 1]\n        item = dict(fold=fold, rows=len(iv), tied_rows=int(fold_ties.sum()),\n            tie_groups=int(len(fold_ties)), largest_tie=int(fold_ties.max()) if len(fold_ties) else 1)\n        per_fold.append(item)\n    return dict(rows=len(development), tied_rows=int(tied.sum()),\n        tied_fraction=float(tied.sum() / len(development)), tie_groups=int(len(tied)),\n        largest_tie=int(tied.max()) if len(tied) else 1, per_fold=per_fold)\n\n\ndef gpu_primary_gate(y, candidate, reference, dev, cv):\n    candidate_auc = float(roc_auc_score(y[dev], candidate[dev]))\n    reference_auc = float(roc_auc_score(y[dev], reference[dev]))\n    candidate_folds = fold_auc_scores(y, candidate, cv)\n    reference_folds = fold_auc_scores(y, reference, cv)\n    wins = sum(left > right + 1e-6 for left, right in zip(candidate_folds, reference_folds))\n    return dict(allowed=candidate_auc > reference_auc + 1e-6 and wins >= 3,\n        auc=candidate_auc, baseline_auc=reference_auc, fold_wins=wins,\n        fold_auc=candidate_folds, baseline_fold_auc=reference_folds)\n\n\ndef freeze(args):\n    run, cfg, tr, te, sub, id_col, cols, y, (dev, sealed, cv, outer) = context(args)\n    if (run / \'frozen.json\').exists():\n        print(\'Already frozen; selection retained.\', flush=True)\n        return\n    if (run / \'SEALED_OPENED.json\').exists():\n        raise RuntimeError(\'Holdout already opened; cannot select candidates again.\')\n    baseline = load_candidate(run, \'main\', y, dev, sealed)\n    selected = [baseline]\n    tie_pool = [baseline]\n    if args.domain_compare:\n        domain = load_candidate(run, \'domain\', y, dev, sealed)\n        selected.append(domain)\n        tie_pool.append(domain)\n    manifest_path = run / \'promoted.json\'\n    manifest = p.read_json(manifest_path) if manifest_path.exists() else {\'lanes\': {}}\n    # Only the latest explicit promotion set can be selected.\n    for lane, limit in [(\'lgb\', 3), (\'xgb\', 2), (\'cat\', 1), (\'realmlp\', 1)]:\n        if not any((run / \'candidates\').glob(f\'{lane}_*.json\')):\n            continue\n        if lane not in manifest.get(\'lanes\', {}):\n            raise ValueError(f\'Missing promoted.json entry for {lane}; rerun search before freeze.\')\n        names = manifest[\'lanes\'][lane]\n        if lane in (\'cat\', \'realmlp\'):\n            complete = set(names)\n        else:\n            study = optuna.load_study(study_name=lane,\n                storage=f"sqlite:///{(run / \'optuna.db\').resolve().as_posix()}")\n            complete = {f\'{lane}_{trial.number}\' for trial in study.trials\n                        if trial.state == optuna.trial.TrialState.COMPLETE}\n        invalid = [name for name in names if name not in complete\n                   or not (run / \'candidates\' / f\'{name}.json\').exists()\n                   or p.read_json(run / \'candidates\' / f\'{name}.json\').get(\'stage\') != \'full\']\n        if invalid:\n            raise ValueError(f\'Invalid promoted candidates for {lane}: {invalid}\')\n        records = [load_candidate(run, name, y, dev, sealed) for name in names]\n        if not records:\n            continue\n        records.sort(key=lambda pair: pair[0][\'dev_auc\'], reverse=True)\n        tie_pool.extend(records)\n        kept = 0\n        for record, oof in records:\n            if record[\'dev_auc\'] < records[0][0][\'dev_auc\'] - args.auc_window or kept >= limit:\n                continue\n            # Check both Pearson and rank correlation; retain weaker-but-different families.\n            redundant = any(np.corrcoef(oof[dev], other[dev])[0, 1] > args.max_corr and\n                np.corrcoef(rankdata(oof[dev]), rankdata(other[dev]))[0, 1] > args.max_corr\n                for _, other in selected)\n            gpu_replacement = (record.get(\'device\') == \'cuda\' and\n                gpu_primary_gate(y, oof, baseline[1], dev, cv)[\'allowed\'])\n            if not redundant or gpu_replacement:\n                selected.append((record, oof))\n                kept += 1\n    gates = {record[\'name\']: gpu_primary_gate(y, oof, baseline[1], dev, cv)\n             for record, oof in selected if record.get(\'device\') == \'cuda\'}\n    eligible = [(record, oof) for record, oof in selected\n                if gates.get(record[\'name\'], {}).get(\'allowed\')]\n    primary = max(eligible, key=lambda pair: pair[0][\'dev_auc\']) if eligible else baseline\n    selected = [primary] + [pair for pair in selected if pair[0][\'name\'] != primary[0][\'name\']]\n    p.write_json(run / \'gpu_primary_selection.json\', dict(cpu_baseline=baseline[0][\'name\'],\n        chosen_primary=primary[0][\'name\'], required_fold_wins=3, candidates=gates))\n    seed_diagnostic = dict(enabled=False, accepted=False, reason=\'Primary is not a GPU model or one seed was requested.\')\n    final_seeds = list(getattr(args, \'final_seeds\', [cfg.get(\'seed\', 2026)]))\n    if primary[0].get(\'device\') == \'cuda\' and len(final_seeds) > 1:\n        averaged_candidate = dict(primary[0], name=primary[0][\'name\'] + \'_seedavg\',\n                                  model_seeds=final_seeds)\n        evaluate(args, cfg, averaged_candidate, requested=list(range(4)), stage=\'full\')\n        averaged = load_candidate(run, averaged_candidate[\'name\'], y, dev, sealed)\n        gate = gpu_primary_gate(y, averaged[1], primary[1], dev, cv)\n        seed_diagnostic = dict(enabled=True, accepted=gate[\'allowed\'], seeds=final_seeds,\n            base_candidate=primary[0][\'name\'], averaged_candidate=averaged[0][\'name\'], **gate)\n        if gate[\'allowed\']:\n            selected[0] = averaged\n            primary = averaged\n            tie_pool.append(averaged)\n    p.write_json(run / \'seed_average_comparison.json\', seed_diagnostic)\n    matrix = np.column_stack([oof for _, oof in selected])\n    rank_matrix = rank_oof_matrix(matrix, cv)\n    best = {}\n    for mode, values in [(\'probability\', matrix), (\'rank\', rank_matrix)]:\n        narrow_cv = crossfit_blend(y, values, cv, .10)\n        wide_cv = crossfit_blend(y, values, cv, .30)\n        wide_allowed, wide_wins = improvement_gate(narrow_cv, wide_cv)\n        cap = .30 if wide_allowed else .10\n        score, weights = best_blend(y, values, dev, cap)\n        winner = dict(auc=score, weights=weights,\n            fold_auc=fold_auc_scores(y, values @ weights, cv), cap=cap,\n            crossfit_auc=(wide_cv if wide_allowed else narrow_cv)[\'auc\'],\n            crossfit_fold_auc=(wide_cv if wide_allowed else narrow_cv)[\'fold_auc\'],\n            crossfit_fold_weights=(wide_cv if wide_allowed else narrow_cv)[\'fold_weights\'],\n            wide_allowed=wide_allowed, wide_fold_wins=wide_wins,\n            narrow_crossfit_auc=narrow_cv[\'auc\'], narrow_crossfit_fold_auc=narrow_cv[\'fold_auc\'],\n            wide_crossfit_auc=wide_cv[\'auc\'], wide_crossfit_fold_auc=wide_cv[\'fold_auc\'])\n        best[mode] = winner\n    allowed, rank_wins = rank_blend_allowed(best[\'probability\'][\'crossfit_auc\'],\n        best[\'rank\'][\'crossfit_auc\'], best[\'probability\'][\'crossfit_fold_auc\'],\n        best[\'rank\'][\'crossfit_fold_auc\'])\n    mode = \'rank\' if allowed else \'probability\'\n    winner = best[mode]\n    blend_matrix = rank_matrix if mode == \'rank\' else matrix\n    base_oof = blend_matrix @ winner[\'weights\']\n    tie_trials = []\n    for record, oof in tie_pool:\n        prediction = lexicographic_oof(base_oof, oof, cv)\n        score = float(roc_auc_score(y[dev], prediction[dev]))\n        scores = fold_auc_scores(y, prediction, cv)\n        accepted, wins = tie_break_allowed(winner[\'auc\'], score, winner[\'fold_auc\'], scores)\n        tie_trials.append(dict(candidate=record, auc=score, fold_auc=scores,\n                               fold_wins=wins, accepted=accepted))\n    accepted = [trial for trial in tie_trials if trial[\'accepted\']]\n    tie_winner = max(accepted, key=lambda trial: trial[\'auc\']) if accepted else None\n    tie_breaker = None if tie_winner is None else dict(candidate=tie_winner[\'candidate\'],\n        auc=tie_winner[\'auc\'], fold_auc=tie_winner[\'fold_auc\'], fold_wins=tie_winner[\'fold_wins\'])\n    p.write_json(run / \'tie_break_comparison.json\', dict(\n        base_auc=winner[\'auc\'], base_fold_auc=winner[\'fold_auc\'],\n        tie_statistics=fold_tie_statistics(base_oof, cv), required_fold_wins=3,\n        candidates=[dict(name=trial[\'candidate\'][\'name\'], auc=trial[\'auc\'],\n            fold_auc=trial[\'fold_auc\'], fold_wins=trial[\'fold_wins\'], accepted=trial[\'accepted\'])\n            for trial in tie_trials],\n        chosen=None if tie_winner is None else tie_winner[\'candidate\'][\'name\']))\n    p.write_json(run / \'blend_comparison.json\', dict(\n        candidates=[c[\'name\'] for c, _ in selected],\n        probability=dict(auc=best[\'probability\'][\'auc\'], weights=best[\'probability\'][\'weights\'].tolist(),\n                         fold_auc=best[\'probability\'][\'fold_auc\'], cap=best[\'probability\'][\'cap\'],\n                         crossfit_auc=best[\'probability\'][\'crossfit_auc\'],\n                         crossfit_fold_auc=best[\'probability\'][\'crossfit_fold_auc\'],\n                         crossfit_fold_weights=best[\'probability\'][\'crossfit_fold_weights\'],\n                         wide_allowed=best[\'probability\'][\'wide_allowed\'],\n                         wide_fold_wins=best[\'probability\'][\'wide_fold_wins\'],\n                         narrow_crossfit_auc=best[\'probability\'][\'narrow_crossfit_auc\'],\n                         wide_crossfit_auc=best[\'probability\'][\'wide_crossfit_auc\']),\n        rank=dict(auc=best[\'rank\'][\'auc\'], weights=best[\'rank\'][\'weights\'].tolist(),\n                  fold_auc=best[\'rank\'][\'fold_auc\'], cap=best[\'rank\'][\'cap\'],\n                  crossfit_auc=best[\'rank\'][\'crossfit_auc\'],\n                  crossfit_fold_auc=best[\'rank\'][\'crossfit_fold_auc\'],\n                  crossfit_fold_weights=best[\'rank\'][\'crossfit_fold_weights\'],\n                  wide_allowed=best[\'rank\'][\'wide_allowed\'],\n                  wide_fold_wins=best[\'rank\'][\'wide_fold_wins\'],\n                  narrow_crossfit_auc=best[\'rank\'][\'narrow_crossfit_auc\'],\n                  wide_crossfit_auc=best[\'rank\'][\'wide_crossfit_auc\']),\n        rank_fold_wins=rank_wins, rank_required_fold_wins=3, chosen_mode=mode))\n    p.write_json(run / \'candidate_correlation.json\', dict(names=[c[\'name\'] for c, _ in selected],\n        pearson=np.nan_to_num(np.atleast_2d(np.corrcoef(matrix[dev], rowvar=False)), nan=1).tolist()))\n    chosen = [(c, float(w)) for (c, _), w in zip(selected, winner[\'weights\']) if w > 0]\n    final_auc = tie_winner[\'auc\'] if tie_winner is not None else winner[\'auc\']\n    final_fold_auc = tie_winner[\'fold_auc\'] if tie_winner is not None else winner[\'fold_auc\']\n    p.write_json(run / \'frozen.json\', dict(candidates=[c for c, _ in chosen], weights=[w for _, w in chosen],\n        mode=mode, tie_breaker=tie_breaker, development_selection_auc=final_auc,\n        development_fold_auc=final_fold_auc, rank_fold_wins=rank_wins,\n        primary=primary[0][\'name\'], baseline=baseline[0],\n        config_sha256=p.digest(run / \'config.json\'),\n        note=\'Weights selected only on development OOF. Selection-biased; no Public LB tuning.\'))\n    tie_name = None if tie_winner is None else tie_winner[\'candidate\'][\'name\']\n    print(f\'Frozen {mode} blend; tie_breaker={tie_name}; dev AUC={final_auc:.7f}; \'\n          f\'weights={[w for _, w in chosen]}\', flush=True)\n\n\ndef frozen_config(run):\n    frozen = p.read_json(run / \'frozen.json\')\n    if frozen[\'config_sha256\'] != p.digest(run / \'config.json\'):\n        raise ValueError(\'Frozen configuration changed\')\n    mark = run / \'SEALED_OPENED.json\'\n    if mark.exists() and p.read_json(mark)[\'frozen_sha256\'] != p.digest(run / \'frozen.json\'):\n        raise ValueError(\'Frozen selection changed after holdout was opened\')\n    return frozen\n', encoding='utf-8')
(PACKAGE / 'models.py').write_text('"""Model construction, fitting, backend selection, and prediction."""\nimport numpy as np\n\nfrom .config import LGB_CUDA_BIN_LIMIT\n\n\ndef model_frame(x, family):\n    if family not in ("cat", "realmlp"):\n        return x\n    x = x.copy()\n    if family == "cat":\n        for c in x.select_dtypes(include="category"):\n            x[c] = x[c].astype(int).astype(str)\n    return x\n\n\ndef realmlp_stop_epoch(model, fallback):\n    values = getattr(model, "fit_params_", {}).get("stop_epoch", {})\n    if isinstance(values, dict):\n        values = list(values.values())\n    values = np.asarray(values if np.size(values) else [fallback], dtype=float).reshape(-1)\n    values = values[np.isfinite(values)]\n    return max(1, int(np.median(values))) if len(values) else max(1, int(fallback))\n\n\ndef lgb_effective_device(params, cfg):\n    requested = cfg.get("lgb_device", "cpu")\n    if requested == "cuda" and params.get("max_bin", 255) > LGB_CUDA_BIN_LIMIT:\n        return "cpu"\n    return requested\n\n\ndef fit_model(family, params, x, y, valid, seed, cfg, rounds=None):\n    n = int(rounds or cfg["max_rounds"])\n    x = model_frame(x, family)\n    valid = None if valid is None else (model_frame(valid[0], family), valid[1])\n    stop = cfg["early_stopping"]\n    if family == "xgb":\n        from xgboost import XGBClassifier\n        model = XGBClassifier(**params, n_estimators=n, objective="binary:logistic", eval_metric="auc",\n            tree_method="hist", enable_categorical=True, device=cfg["xgb_device"],\n            n_jobs=cfg["threads"], random_state=seed, early_stopping_rounds=stop if valid else None)\n        model.fit(x, y, eval_set=[valid] if valid else None, verbose=False)\n        best = model.best_iteration + 1 if valid else n\n    elif family == "lgb":\n        import lightgbm as lgb\n        device = lgb_effective_device(params, cfg)\n        if device != cfg.get("lgb_device", "cpu"):\n            print(f"  LightGBM max_bin={params[\'max_bin\']}: using CPU to avoid the observed CUDA crash; bins unchanged", flush=True)\n        backend = (dict(deterministic=True, force_col_wise=True) if device == "cpu" else\n                   dict(device_type="cuda", gpu_device_id=cfg.get("lgb_gpu_id", 0), num_gpu=1))\n        if device not in ("cpu", "cuda"):\n            raise ValueError("LightGBM device must be cpu or cuda")\n        model = lgb.LGBMClassifier(**params, n_estimators=n, objective="binary", metric="auc",\n            n_jobs=cfg["threads"], random_state=seed, verbosity=-1, **backend)\n        model.fit(x, y, eval_set=[valid] if valid else None,\n            callbacks=[lgb.early_stopping(stop, verbose=False)] if valid else [])\n        best = model.best_iteration_ if valid else n\n    elif family == "cat":\n        from catboost import CatBoostClassifier\n        cats = list(x.select_dtypes(include=["object", "string"]))\n        cat_device = cfg.get("cat_device", "cpu")\n        if cat_device not in ("cpu", "cuda"):\n            raise ValueError("CatBoost device must be cpu or cuda")\n        backend = (dict(task_type="GPU", devices=str(cfg.get("cat_gpu_id", 0)))\n                   if cat_device == "cuda" else dict(task_type="CPU"))\n        model = CatBoostClassifier(**params, iterations=n, loss_function="Logloss", eval_metric="AUC",\n            thread_count=cfg["threads"], random_seed=seed, allow_writing_files=False, verbose=False, **backend)\n        model.fit(x, y, cat_features=cats, eval_set=valid, use_best_model=bool(valid),\n            early_stopping_rounds=stop if valid else None, verbose=False)\n        best = model.get_best_iteration() + 1 if valid else n\n    elif family == "realmlp":\n        from pytabkit import RealMLP_TD_Classifier\n        options = dict(params)\n        epochs = int(options.pop("n_epochs", min(n, 256))) if rounds is None else n\n        options.update(device="cuda:0" if cfg.get("realmlp_device") == "cuda" else "cpu",\n            random_state=seed, n_threads=cfg["threads"], verbosity=0,\n            val_metric_name="1-auc_ovr", use_ls=False)\n        if valid is None:\n            options.update(stop_epoch=epochs, val_fraction=0.0)\n        else:\n            options["n_epochs"] = epochs\n        model = RealMLP_TD_Classifier(**options)\n        if valid is None:\n            model.fit(x, y)\n            best = epochs\n        else:\n            model.fit(x, y, valid[0], valid[1])\n            best = realmlp_stop_epoch(model, epochs)\n    else:\n        raise ValueError(f"Unknown model family: {family}")\n    return model, max(1, int(best))\n\n\ndef predict(model, x, family):\n    p = model.predict_proba(model_frame(x, family))[:, 1]\n    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():\n        raise ValueError("Invalid probabilities")\n    return p\n\n\ndef defaults(family):\n    if family == "xgb":\n        return dict(max_depth=5, min_child_weight=8., gamma=0.1, reg_alpha=0.01,\n            reg_lambda=8., subsample=0.85, colsample_bytree=0.85, learning_rate=0.04)\n    if family == "lgb":\n        return dict(max_depth=6, num_leaves=31, min_child_samples=80, reg_alpha=0.01,\n            reg_lambda=8., subsample=0.85, subsample_freq=1, colsample_bytree=0.85, learning_rate=0.04, max_bin=255)\n    if family == "cat":\n        return dict(depth=6, learning_rate=0.04, l2_leaf_reg=8., random_strength=1.,\n            bootstrap_type="Bayesian", bagging_temperature=1.)\n    if family == "realmlp":\n        return dict(n_epochs=128)\n    raise ValueError(f"Unknown model family: {family}")\n', encoding='utf-8')
(PACKAGE / 'split.py').write_text('"""Leak-free development and sealed split definitions."""\nimport numpy as np\nfrom sklearn.model_selection import StratifiedKFold\n\nfrom .config import SEED\n\n\ndef split_plan(y, seed=SEED):\n    """Sealed fold never enters any development fold\'s training indices."""\n    outer = np.full(len(y), -1, dtype=int)\n    for fold, (_, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=seed).split(np.zeros(len(y)), y)):\n        outer[va] = fold\n    dev, sealed = np.flatnonzero(outer != 4), np.flatnonzero(outer == 4)\n    cv = [(dev[tr], dev[va]) for tr, va in StratifiedKFold(4, shuffle=True, random_state=seed + 1).split(dev, y[dev])]\n    return dev, sealed, cv, outer\n', encoding='utf-8')
(PACKAGE / 'tuning.py').write_text('"""Candidate tuning and cached CPU/T4 fold execution."""\nfrom __future__ import annotations\n\nimport gc\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport traceback\n\nimport joblib\nimport numpy as np\nimport optuna\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\n\nimport prototype as p\nfrom .artifact import (checked_prediction, key, read_json, write_json,\n                       write_runtime_profile, write_screen_diagnostics)\n\n\ndef worker_entrypoint():\n    return Path(__file__).resolve().parent.parent / \'diversity.py\'\n\n\ndef context(args):\n    run = Path(args.run)\n    source_root = Path(__file__).resolve().parent.parent\n    source_files = [source_root / name for name in (\'prototype.py\', \'diversity.py\', \'gpu_setup.py\')]\n    source_files.extend(sorted((source_root / \'s6e9\').glob(\'*.py\')))\n    settings = dict(seed=args.seed, threads=args.threads, gpu_ids=args.gpu_ids,\n        parallel_folds=2, max_rounds=args.max_rounds,\n        early_stopping=args.early_stopping, auc_window=args.auc_window, max_corr=args.max_corr,\n        domain_compare=args.domain_compare, cat_compare=args.cat_compare,\n        realmlp_compare=args.realmlp_compare, final_seeds=args.final_seeds,\n        screen_folds=args.screen_folds, promote_trials=args.promote_trials,\n        hashes={f: p.digest(Path(args.data) / f) for f in (\'train.csv\', \'test.csv\', \'sample_submission.csv\')},\n        sources={path.relative_to(source_root).as_posix(): p.digest(path) for path in source_files},\n        versions={name: importlib.metadata.version(name) for name in\n            (\'numpy\', \'pandas\', \'scikit-learn\', \'lightgbm\', \'xgboost\', \'catboost\', \'optuna\',\n             \'joblib\', \'scipy\') + ((\'pytabkit\',) if args.realmlp_compare else ())})\n    path = run / \'config.json\'\n    if path.exists():\n        if p.read_json(path) != settings:\n            raise ValueError(\'Data, implementation or settings changed. Choose a new v5 RUN; old results are retained.\')\n    elif args.command == \'search\':\n        p.write_json(path, settings)\n    else:\n        raise ValueError(\'Run search first with the same settings.\')\n    tr, te, sub, id_col, cols = p.load_data(args.data)\n    y = tr[p.TARGET].to_numpy(dtype=int)\n    split = p.split_plan(y, args.seed)\n    data_path = run / \'cache\' / \'data\' / (key(dict(hashes=settings[\'hashes\'], seed=args.seed,\n        versions=settings[\'versions\'], source=settings[\'sources\'][\'prototype.py\'])) + \'.joblib\')\n    meta_path = data_path.with_suffix(\'.json\')\n    if data_path.exists() and meta_path.exists():\n        if p.read_json(meta_path)[\'sha256\'] != p.digest(data_path):\n            raise ValueError(f\'Corrupt worker data cache: {data_path}\')\n    else:\n        data_path.parent.mkdir(parents=True, exist_ok=True)\n        temporary = data_path.with_suffix(\'.tmp\')\n        joblib.dump((tr, te, id_col, cols, y, split), temporary, compress=0)\n        temporary.replace(data_path)\n        p.write_json(meta_path, dict(sha256=p.digest(data_path), bytes=data_path.stat().st_size))\n    args._data = (tr, te, sub, id_col, cols, y, split)\n    return run, dict(settings, _data_cache=str(data_path.resolve())), tr, te, sub, id_col, cols, y, split\n\n\ndef anchor():\n    params = dict(p.defaults(\'lgb\'), max_depth=5, num_leaves=31, min_child_samples=20,\n        colsample_bytree=.5, reg_alpha=.07, reg_lambda=2., max_bin=511)\n    return dict(name=\'main\', lane=\'main\', family=\'lgb\', variant=\'multiscale_dual\',\n                params=params, device=\'cpu\')\n\n\ndef cat_anchor():\n    return dict(name=\'cat_fixed\', lane=\'cat\', family=\'cat\', variant=\'multiscale_dual\',\n                params=p.defaults(\'cat\'), device=\'cuda\')\n\n\ndef domain_anchor():\n    params = dict(anchor()[\'params\'], max_bin=p.LGB_CUDA_BIN_LIMIT)\n    return dict(name=\'domain\', lane=\'domain\', family=\'lgb\', variant=\'multiscale_domain\',\n                params=params, device=\'cuda\')\n\n\ndef realmlp_anchor():\n    return dict(name=\'realmlp_fixed\', lane=\'realmlp\', family=\'realmlp\', variant=\'raw\',\n                params=p.defaults(\'realmlp\'), device=\'cuda\')\n\n\ndef suggest(trial, lane):\n    if lane == \'lgb\':\n        depth = trial.suggest_int(\'max_depth\', 4, 7)\n        params = dict(max_depth=depth, num_leaves=min(2 ** depth, trial.suggest_int(\'num_leaves\', 20, 64)),\n            min_child_samples=trial.suggest_int(\'min_child_samples\', 10, 150),\n            reg_alpha=trial.suggest_float(\'reg_alpha\', 1e-4, 2, log=True),\n            reg_lambda=trial.suggest_float(\'reg_lambda\', .2, 10, log=True),\n            subsample=trial.suggest_float(\'subsample\', .7, 1), subsample_freq=1,\n            colsample_bytree=trial.suggest_float(\'colsample_bytree\', .35, .8),\n            learning_rate=trial.suggest_float(\'learning_rate\', .015, .06, log=True),\n            max_bin=trial.suggest_categorical(\'max_bin\', [63, 127, 255]))\n    elif lane == \'xgb\':\n        params = dict(max_depth=trial.suggest_int(\'max_depth\', 4, 8),\n            min_child_weight=trial.suggest_float(\'min_child_weight\', 2, 30, log=True),\n            gamma=trial.suggest_float(\'gamma\', 0, 2),\n            reg_alpha=trial.suggest_float(\'reg_alpha\', 1e-4, 3, log=True),\n            reg_lambda=trial.suggest_float(\'reg_lambda\', .5, 15, log=True),\n            subsample=trial.suggest_float(\'subsample\', .7, .95),\n            colsample_bytree=trial.suggest_float(\'colsample_bytree\', .4, .9),\n            learning_rate=trial.suggest_float(\'learning_rate\', .02, .06, log=True))\n    else:\n        raise ValueError(f\'Unknown search lane: {lane}\')\n    return dict(name=f\'{lane}_{trial.number}\', lane=lane, family=lane,\n        variant=\'multiscale_dual\', params=params, device=\'cuda\')\n\n\ndef worker(path, data_state=None):\n    """Load binary data once and process all folds assigned to this device."""\n    job = p.read_json(path)\n    cfg, candidate = job[\'config\'], job[\'candidate\']\n    if data_state is None:\n        data = joblib.load(cfg[\'_data_cache\'])\n    else:\n        cache_path = str(Path(cfg[\'_data_cache\']).resolve())\n        if data_state.get(\'path\') != cache_path:\n            data_state.clear()\n            data_state.update(path=cache_path, value=joblib.load(cache_path))\n        data = data_state[\'value\']\n    tr, te, _, cols, y, (_, _, cv, outer) = data\n    phase = job[\'phase\']\n    for fold in job[\'folds\']:\n        started = time.perf_counter()\n        if phase == \'dev\':\n            it, iv = cv[fold]\n            xp = tr[cols].iloc[iv]\n        else:\n            it, iv = np.flatnonzero(outer != fold), np.flatnonzero(outer == fold)\n            xp = pd.concat([tr[cols].iloc[iv], te[cols]], ignore_index=True)\n        xt = tr[cols].iloc[it]\n        identity = dict(hashes=cfg[\'hashes\'], sources=cfg[\'sources\'], versions=cfg[\'versions\'],\n            seed=cfg[\'seed\'], variant=candidate[\'variant\'], phase=phase, fold=fold)\n        cache = Path(job[\'run\']) / \'cache\' / \'features\' / (key(identity) + \'.joblib\')\n        meta_path = cache.with_suffix(\'.json\')\n        cache_hit = cache.exists() and meta_path.exists()\n        if cache_hit:\n            if p.read_json(meta_path)[\'sha256\'] != p.digest(cache):\n                raise ValueError(f\'Corrupt feature cache: {cache}\')\n            train_x, pred_x = joblib.load(cache)\n            print(f\'Reused features: {candidate["variant"]}/{phase}/{fold}\', flush=True)\n        else:\n            fe = p.make_features(candidate[\'variant\'], cfg[\'seed\'])\n            train_x, pred_x = fe.fit_transform(xt, y[it]), fe.transform(xp)\n            cache.parent.mkdir(parents=True, exist_ok=True)\n            temporary = cache.with_suffix(\'.tmp\')\n            joblib.dump((train_x, pred_x), temporary, compress=0)\n            temporary.replace(cache)\n            p.write_json(meta_path, dict(sha256=p.digest(cache), compression=0, bytes=cache.stat().st_size))\n        valid = (pred_x, y[iv]) if phase == \'dev\' else None\n        fixed = candidate.get(\'fixed_rounds\') if phase != \'dev\' else None\n        model_seed = candidate.get(\'model_seed\', cfg[\'seed\'])\n        backend = dict(max_rounds=cfg[\'max_rounds\'], early_stopping=cfg[\'early_stopping\'],\n            threads=job[\'threads\'], lgb_device=candidate[\'device\'], lgb_gpu_id=0,\n            xgb_device=\'cuda:0\' if candidate[\'device\'] == \'cuda\' else \'cpu\',\n            cat_device=candidate[\'device\'], cat_gpu_id=0,\n            realmlp_device=candidate[\'device\'])\n        prepared_at = time.perf_counter()\n        model, rounds = p.fit_model(candidate[\'family\'], candidate[\'params\'], train_x, y[it],\n            valid, model_seed, backend, fixed)\n        if candidate[\'family\'] == \'xgb\' and candidate[\'device\'] == \'cuda\':\n            actual = json.loads(model.get_booster().save_config())[\'learner\'][\'generic_param\'][\'device\']\n            if not actual.startswith(\'cuda\'):\n                raise RuntimeError(f\'XGBoost did not use the requested GPU: {actual}\')\n        if candidate[\'family\'] == \'cat\' and candidate[\'device\'] == \'cuda\':\n            actual = str(model.get_param(\'task_type\') or \'\').upper()\n            if actual != \'GPU\':\n                raise RuntimeError(f\'CatBoost did not use the requested GPU: {actual}\')\n        if candidate[\'family\'] == \'realmlp\' and candidate[\'device\'] == \'cuda\':\n            import torch\n            if not torch.cuda.is_available() or torch.cuda.device_count() != 1:\n                raise RuntimeError(\'RealMLP worker does not see its isolated T4.\')\n        prediction = p.predict(model, pred_x, candidate[\'family\'])\n        output = Path(job[\'outputs\'][str(fold)])\n        temporary = output.with_suffix(\'.tmp\')\n        with temporary.open(\'wb\') as stream:\n            np.save(stream, prediction, allow_pickle=False)\n        temporary.replace(output)\n        p.write_json(output.with_suffix(\'.json\'), dict(sha256=p.digest(output), rounds=max(1, int(rounds)),\n            candidate=candidate[\'name\'], family=candidate[\'family\'], device=candidate[\'device\'],\n            phase=phase, fold=fold, model_seed=model_seed,\n            validation_rows=len(iv), test_rows=0 if phase == \'dev\' else len(te), feature_cache_hit=cache_hit,\n            prepare_seconds=prepared_at-started, fit_predict_seconds=time.perf_counter()-prepared_at,\n            feature_cache_bytes=cache.stat().st_size))\n\n\ndef worker_loop():\n    """Keep one interpreter and binary data cache alive for one isolated GPU."""\n    data_state = {}\n    for line in sys.stdin:\n        job_path = line.strip()\n        if not job_path:\n            continue\n        status = None\n        try:\n            job = p.read_json(job_path)\n            status = Path(job[\'status\'])\n            worker(job_path, data_state)\n            p.write_json(status, {\'ok\': True})\n        except BaseException:\n            error = traceback.format_exc()\n            print(error, file=sys.stderr, flush=True)\n            if status is not None:\n                p.write_json(status, {\'ok\': False, \'error\': error})\n        finally:\n            gc.collect()\n\n\nclass GpuWorkerPool:\n    """One persistent child per T4; jobs remain isolated by CUDA visibility."""\n    def __init__(self):\n        self.processes = {}\n\n    def _start(self, slot, selector, threads, log_path):\n        env = dict(os.environ, OMP_NUM_THREADS=str(threads), OPENBLAS_NUM_THREADS=str(threads),\n            PYTHONFAULTHANDLER=\'1\', PYTHONUNBUFFERED=\'1\', CUDA_VISIBLE_DEVICES=str(selector))\n        log = log_path.open(\'a\', encoding=\'utf8\')\n        try:\n            proc = subprocess.Popen([sys.executable, \'-X\', \'faulthandler\', \'-u\', str(worker_entrypoint()),\n                \'worker-loop\'], stdin=subprocess.PIPE, stdout=log, stderr=subprocess.STDOUT,\n                env=env, text=True, bufsize=1)\n        except BaseException:\n            log.close()\n            raise\n        self.processes[slot] = dict(selector=str(selector), threads=threads, proc=proc,\n            log=log, log_path=log_path)\n        return self.processes[slot]\n\n    def submit(self, slot, selector, threads, job_path, status_path, log_path):\n        item = self.processes.get(slot)\n        if item is not None and (item[\'selector\'] != str(selector) or item[\'threads\'] != threads\n                                 or item[\'proc\'].poll() is not None):\n            self._stop(item)\n            self.processes.pop(slot, None)\n            item = None\n        if item is None:\n            item = self._start(slot, selector, threads, log_path)\n        status_path.unlink(missing_ok=True)\n        item[\'proc\'].stdin.write(str(Path(job_path).resolve()) + \'\\n\')\n        item[\'proc\'].stdin.flush()\n        return dict(item, status_path=status_path)\n\n    @staticmethod\n    def _tail(path):\n        return path.read_text(encoding=\'utf8\', errors=\'replace\')[-5000:] if path.exists() else \'\'\n\n    def wait(self, tasks):\n        pending = list(tasks)\n        while pending:\n            for task in pending[:]:\n                if task[\'status_path\'].exists():\n                    status = p.read_json(task[\'status_path\'])\n                    if not status.get(\'ok\'):\n                        raise RuntimeError(f\'GPU worker failed. Log: {task["log_path"]}\\n\'\n                            f\'{status.get("error", self._tail(task["log_path"]))}\')\n                    pending.remove(task)\n                elif task[\'proc\'].poll() is not None:\n                    raise RuntimeError(f\'GPU worker exited ({task["proc"].poll()}). Log: \'\n                        f\'{task["log_path"]}\\n{self._tail(task["log_path"])}\')\n            if pending:\n                time.sleep(.05)\n\n    @staticmethod\n    def _stop(item):\n        proc = item[\'proc\']\n        try:\n            if proc.poll() is None:\n                proc.stdin.close()\n                try:\n                    proc.wait(timeout=10)\n                except subprocess.TimeoutExpired:\n                    proc.terminate()\n                    proc.wait()\n        finally:\n            item[\'log\'].close()\n\n    def shutdown(self):\n        for item in list(self.processes.values()):\n            self._stop(item)\n        self.processes.clear()\n\n\n_GPU_POOL = None\n\n\ndef gpu_pool():\n    global _GPU_POOL\n    if _GPU_POOL is None:\n        _GPU_POOL = GpuWorkerPool()\n    return _GPU_POOL\n\n\ndef shutdown_gpu_workers():\n    global _GPU_POOL\n    if _GPU_POOL is not None:\n        _GPU_POOL.shutdown()\n        _GPU_POOL = None\n\n\ndef single_seed_folds(args, cfg, candidate, phase, requested):\n    run = Path(args.run)\n    cache = run / \'cache\' / \'predictions\'\n    cache.mkdir(parents=True, exist_ok=True)\n    jobs = run / \'cache\' / \'jobs\'\n    jobs.mkdir(parents=True, exist_ok=True)\n    results = {}\n    y, (_, _, cv, _) = args._data[5], args._data[6]\n    spec = {k: candidate[k] for k in (\'family\', \'variant\', \'params\', \'device\')}\n    if \'model_seed\' in candidate:\n        spec[\'model_seed\'] = candidate[\'model_seed\']\n    if phase != \'dev\':\n        spec[\'fixed_rounds\'] = candidate[\'fixed_rounds\']\n    threads = args.threads if candidate[\'device\'] == \'cpu\' else max(1, args.threads // 2)\n    pending, outputs = [], {}\n    for fold in requested:\n        tag = key(dict(config=cfg, candidate=spec, phase=phase, fold=fold, threads=threads))\n        output = cache / f\'{tag}.npy\'\n        outputs[fold] = output\n        if output.exists() and output.with_suffix(\'.json\').exists():\n            results[fold] = checked_prediction(output)\n            print(f\'Reused predictions: {candidate["name"]}/{phase}/fold {fold}\', flush=True)\n        else:\n            pending.append(fold)\n    groups = ([pending] if candidate[\'device\'] == \'cpu\' else\n              [[fold for fold in pending if fold % 2 == slot] for slot in range(2)])\n    processes, tasks = [], []\n    try:\n        for slot, assigned in enumerate(groups):\n            if not assigned:\n                continue\n            selector = \'\' if candidate[\'device\'] == \'cpu\' else str(args.gpu_ids[slot])\n            tag = key(dict(candidate=spec, phase=phase, folds=assigned, selector=selector, threads=threads))\n            job = jobs / f\'{tag}.json\'\n            status_path = jobs / f\'{tag}.status.json\'\n            p.write_json(job, dict(config=cfg, candidate=candidate, phase=phase, folds=assigned,\n                threads=threads, run=str(run.resolve()), status=str(status_path.resolve()),\n                outputs={str(f): str(outputs[f].resolve()) for f in assigned}))\n            env = dict(os.environ, OMP_NUM_THREADS=str(threads), OPENBLAS_NUM_THREADS=str(threads),\n                PYTHONFAULTHANDLER=\'1\', PYTHONUNBUFFERED=\'1\', CUDA_VISIBLE_DEVICES=selector)\n            log_path = jobs / (f\'gpu_worker_{slot}.log\' if candidate[\'device\'] == \'cuda\' else f\'{tag}.log\')\n            device_label = \'CPU\' if candidate[\'device\'] == \'cpu\' else f\'GPU {selector}\'\n            print(f\'Starting {candidate["name"]}/{phase}/folds {assigned} on {device_label}; log={log_path}\', flush=True)\n            if candidate[\'device\'] == \'cuda\':\n                tasks.append((assigned, gpu_pool().submit(slot, selector, threads, job, status_path, log_path)))\n            else:\n                log = log_path.open(\'w\', encoding=\'utf8\')\n                try:\n                    proc = subprocess.Popen([sys.executable, \'-X\', \'faulthandler\', \'-u\', str(worker_entrypoint()),\n                        \'worker\', \'--job\', str(job.resolve())], stdout=log, stderr=subprocess.STDOUT, env=env)\n                except BaseException:\n                    log.close()\n                    raise\n                processes.append((assigned, proc, log, log_path))\n        if tasks:\n            gpu_pool().wait([task for _, task in tasks])\n            for assigned, _ in tasks:\n                for fold in assigned:\n                    results[fold] = checked_prediction(outputs[fold])\n        for assigned, proc, log, log_path in processes:\n            code = proc.wait()\n            log.close()\n            if code:\n                tail = log_path.read_text(encoding=\'utf8\', errors=\'replace\')[-5000:]\n                raise RuntimeError(f\'{candidate["name"]}/{phase}/folds {assigned} failed ({code}). Log: {log_path}\\n{tail}\')\n            for fold in assigned:\n                results[fold] = checked_prediction(outputs[fold])\n    finally:\n        for _, proc, log, _ in processes:\n            if proc.poll() is None:\n                proc.terminate()\n                proc.wait()\n            log.close()\n    for fold in requested:\n        if phase == \'dev\':\n            score = float(roc_auc_score(y[cv[fold][1]], results[fold][0]))\n            print(f\'{candidate["name"]} fold={fold} AUC={score:.7f}\', flush=True)\n    return results\n\n\ndef folds(args, cfg, candidate, phase, requested):\n    seeds = candidate.get(\'model_seeds\')\n    if not seeds:\n        return single_seed_folds(args, cfg, candidate, phase, requested)\n    members = []\n    for seed in seeds:\n        member = {key: value for key, value in candidate.items() if key != \'model_seeds\'}\n        if seed != cfg[\'seed\']:\n            member[\'model_seed\'] = seed\n        members.append(single_seed_folds(args, cfg, member, phase, requested))\n    averaged = {}\n    for fold in requested:\n        predictions = [result[fold][0] for result in members]\n        metadata = dict(members[0][fold][1])\n        metadata.update(rounds=int(np.median([result[fold][1][\'rounds\'] for result in members])),\n                        model_seeds=list(seeds), ensemble_size=len(seeds))\n        averaged[fold] = np.mean(predictions, axis=0), metadata\n    return averaged\n\n\ndef evaluate(args, cfg, candidate, requested=None, stage=\'full\'):\n    requested = list(range(4)) if requested is None else sorted(set(requested))\n    if not requested or any(fold not in range(4) for fold in requested):\n        raise ValueError(\'Development folds must be a non-empty subset of 0..3\')\n    if stage not in (\'screen\', \'full\'):\n        raise ValueError(\'Evaluation stage must be screen or full\')\n    result = folds(args, cfg, candidate, \'dev\', requested)\n    y, (dev, _, cv, _) = args._data[5], args._data[6]\n    oof = np.full(len(y), np.nan)\n    for fold in requested:\n        oof[cv[fold][1]] = result[fold][0]\n    scored = np.concatenate([cv[fold][1] for fold in requested])\n    score = float(roc_auc_score(y[scored], oof[scored]))\n    fold_auc = {str(fold): float(roc_auc_score(y[cv[fold][1]], oof[cv[fold][1]]))\n                for fold in requested}\n    rounds = [result[fold][1][\'rounds\'] for fold in requested]\n    path = Path(args.run) / \'candidates\' / candidate[\'name\']\n    previous = p.read_json(path.with_suffix(\'.json\')) if path.with_suffix(\'.json\').exists() else {}\n    record = dict(candidate, stage=stage,\n        screen_folds=requested if stage == \'screen\' else previous.get(\'screen_folds\'),\n        screen_auc=score if stage == \'screen\' else previous.get(\'screen_auc\'),\n        screen_fold_auc=fold_auc if stage == \'screen\' else previous.get(\'screen_fold_auc\'),\n        dev_auc=score if stage == \'full\' else None,\n        full_fold_auc=fold_auc if stage == \'full\' else None,\n        fixed_rounds=int(np.median(rounds)), rounds=rounds)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    np.save(path.with_suffix(\'.npy\'), oof)\n    record[\'oof_sha256\'] = p.digest(path.with_suffix(\'.npy\'))\n    p.write_json(path.with_suffix(\'.json\'), record)\n    return score\n\n\ndef search(args):\n    run, cfg, *_ = context(args)\n    if (run / \'frozen.json\').exists():\n        print(\'Candidates already frozen; search skipped.\', flush=True)\n        return\n    if (run / \'SEALED_OPENED.json\').exists() or (run / \'sealed_report.json\').exists():\n        raise RuntimeError(\'Holdout already opened; search is disabled for this run.\')\n    if not (run / \'candidates/main.json\').exists():\n        evaluate(args, cfg, anchor())\n    if args.domain_compare and not (run / \'candidates/domain.json\').exists():\n        evaluate(args, cfg, domain_anchor())\n    for lane, budget in [(\'lgb\', args.lgb_trials), (\'xgb\', args.xgb_trials)]:\n        if budget == 0:\n            continue\n        study = optuna.create_study(study_name=lane, direction=\'maximize\',\n            storage=f"sqlite:///{(run / \'optuna.db\').resolve().as_posix()}", load_if_exists=True)\n        # Parent interruption leaves RUNNING; a failed child is marked FAIL by Optuna.\n        # Both cases retry exact parameters so completed fold predictions remain reusable.\n        requeued = {t.user_attrs.get(\'retry_of\') for t in study.trials}\n        for old in study.trials:\n            interrupted = old.state == optuna.trial.TrialState.RUNNING\n            failed = old.state == optuna.trial.TrialState.FAIL and old.user_attrs.get(\'retry_pending\')\n            if (interrupted or failed) and old.number not in requeued:\n                retry = dict(old.system_attrs.get(\'fixed_params\', {}), **old.params)\n                if interrupted:\n                    study.tell(old.number, state=optuna.trial.TrialState.FAIL)\n                study.enqueue_trial(retry, user_attrs={\'retry_of\': old.number})\n        study.sampler = optuna.samplers.TPESampler(seed=args.seed + len(study.trials))\n        done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)\n        def objective(trial):\n            try:\n                return evaluate(args, cfg, suggest(trial, lane),\n                    requested=list(range(args.screen_folds)), stage=\'screen\')\n            except BaseException:\n                trial.set_user_attr(\'retry_pending\', True)\n                raise\n        study.optimize(objective, n_trials=max(0, budget - done))\n        complete = sorted((trial for trial in study.trials\n            if trial.state == optuna.trial.TrialState.COMPLETE and trial.value is not None),\n            key=lambda trial: trial.value, reverse=True)\n        promoted = complete[:min(args.promote_trials, len(complete))]\n        for trial in promoted:\n            path = run / \'candidates\' / f\'{lane}_{trial.number}.json\'\n            record = p.read_json(path)\n            if record.get(\'stage\') == \'full\':\n                continue\n            candidate = {name: record[name]\n                         for name in (\'name\', \'lane\', \'family\', \'variant\', \'params\', \'device\')}\n            print(f\'Promoting {candidate["name"]} to full 4-fold CV\', flush=True)\n            evaluate(args, cfg, candidate, requested=list(range(4)), stage=\'full\')\n        manifest_path = run / \'promoted.json\'\n        manifest = p.read_json(manifest_path) if manifest_path.exists() else {\n            \'screen_folds\': args.screen_folds, \'promote_trials\': args.promote_trials, \'lanes\': {}}\n        manifest[\'lanes\'][lane] = [f\'{lane}_{trial.number}\' for trial in promoted]\n        p.write_json(manifest_path, manifest)\n    if args.cat_compare:\n        candidate = cat_anchor()\n        path = run / \'candidates\' / \'cat_fixed.json\'\n        if not path.exists():\n            evaluate(args, cfg, candidate, requested=list(range(args.screen_folds)), stage=\'screen\')\n        record = p.read_json(path)\n        if record.get(\'stage\') != \'full\':\n            print(\'Promoting cat_fixed to full 4-fold CV\', flush=True)\n            evaluate(args, cfg, candidate, requested=list(range(4)), stage=\'full\')\n        manifest_path = run / \'promoted.json\'\n        manifest = p.read_json(manifest_path) if manifest_path.exists() else {\n            \'screen_folds\': args.screen_folds, \'promote_trials\': args.promote_trials, \'lanes\': {}}\n        manifest[\'lanes\'][\'cat\'] = [\'cat_fixed\']\n        p.write_json(manifest_path, manifest)\n    if getattr(args, \'realmlp_compare\', False):\n        candidate = realmlp_anchor()\n        path = run / \'candidates\' / \'realmlp_fixed.json\'\n        if not path.exists():\n            evaluate(args, cfg, candidate, requested=list(range(args.screen_folds)), stage=\'screen\')\n        record = p.read_json(path)\n        if record.get(\'stage\') != \'full\':\n            print(\'Promoting realmlp_fixed to full 4-fold CV\', flush=True)\n            evaluate(args, cfg, candidate, requested=list(range(4)), stage=\'full\')\n        manifest_path = run / \'promoted.json\'\n        manifest = p.read_json(manifest_path) if manifest_path.exists() else {\n            \'screen_folds\': args.screen_folds, \'promote_trials\': args.promote_trials, \'lanes\': {}}\n        manifest[\'lanes\'][\'realmlp\'] = [\'realmlp_fixed\']\n        p.write_json(manifest_path, manifest)\n    write_screen_diagnostics(run)\n    write_runtime_profile(run)\n    print(\'Search complete. No holdout labels used.\', flush=True)\n', encoding='utf-8')
SCRIPT = Path('/kaggle/working/prototype.py')
SCRIPT.write_text('"""Compatibility CLI for the original S6E9 prototype workflow."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport time\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import train_test_split\n\nfrom s6e9.artifact import digest, read_json, write_json\nfrom s6e9.config import LGB_CUDA_BIN_LIMIT, SEED, TARGET\nfrom s6e9.data import load_data\nfrom s6e9.features import (DomainSignalFeatures, Features, SIGNAL_VARIANTS, SignalFeatures,\n                            augment, domain_features, make_features)\nfrom s6e9.models import (defaults, fit_model, lgb_effective_device, model_frame, predict,\n                          realmlp_stop_epoch)\nfrom s6e9.split import split_plan\n\nRECOVERABLE_CUDA_CODE_HASHES = {"b44e9240ab8c59b4a3b790b42d69d6ed12189322ae91f821c79c98b85609031e"}\n\n\ndef suggest(trial, family):\n    p = defaults(family)\n    p["learning_rate"] = trial.suggest_float("learning_rate", 0.01, 0.1, log=True)\n    if family in ("xgb", "lgb"):\n        p.update(max_depth=trial.suggest_int("max_depth", 3, 9),\n            reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 10, log=True),\n            reg_lambda=trial.suggest_float("reg_lambda", 1e-2, 30, log=True),\n            subsample=trial.suggest_float("subsample", 0.65, 1),\n            colsample_bytree=trial.suggest_float("colsample_bytree", 0.3, 1))\n        if family == "xgb":\n            p.update(min_child_weight=trial.suggest_float("min_child_weight", 1, 30, log=True),\n                     gamma=trial.suggest_float("gamma", 0, 5))\n        else:\n            p.update(num_leaves=min(2 ** p["max_depth"], trial.suggest_int("num_leaves", 15, 127)),\n                     min_child_samples=trial.suggest_int("min_child_samples", 10, 400),\n                     max_bin=trial.suggest_categorical("max_bin", [255, 511, 1023]))\n    else:\n        p.update(depth=trial.suggest_int("depth", 4, 8), l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 30, log=True),\n            random_strength=trial.suggest_float("random_strength", 0.01, 5, log=True),\n            bagging_temperature=trial.suggest_float("bagging_temperature", 0, 5))\n    return p\n\n\ndef context(args, create=False):\n    tr, te, sub, id_col, features = load_data(args.data)\n    run = Path(args.run)\n    cfg_path = run / "config.json"\n    hashes = {name: digest(Path(args.data) / name) for name in ("train.csv", "test.csv", "sample_submission.csv")}\n    code_hash = digest(__file__)\n    if create and not cfg_path.exists():\n        cfg = dict(seed=SEED, max_rounds=args.max_rounds, early_stopping=args.early_stopping,\n            threads=args.threads, xgb_device=args.xgb_device, seeds=args.seeds,\n            lgb_device=args.lgb_device, lgb_gpu_id=args.lgb_gpu_id,\n            lgb_cuda_bin_limit=LGB_CUDA_BIN_LIMIT,\n            holdout_previously_reviewed=args.holdout_already_reviewed, profile=args.profile,\n            hashes=hashes, code_sha256=code_hash, families=args.families,\n            versions={p: importlib.metadata.version(p) for p in ("numpy", "pandas", "scikit-learn", "xgboost", "lightgbm", "catboost", "optuna")},\n            python=platform.python_version())\n        write_json(cfg_path, cfg)\n    cfg = read_json(cfg_path)\n    if cfg["hashes"] != hashes or cfg["code_sha256"] != code_hash:\n        raise ValueError("Data/code changed. Use a fresh run directory; do not reuse a revealed sealed holdout for tuning.")\n    if cfg.get("lgb_cuda_bin_limit") != LGB_CUDA_BIN_LIMIT:\n        raise ValueError("Backend policy changed. Use a new run or the explicit recover-run command.")\n    if create:\n        for key in ("max_rounds", "early_stopping", "threads", "xgb_device", "lgb_device", "lgb_gpu_id", "seeds", "families", "profile"):\n            if getattr(args, key) != cfg[key]:\n                raise ValueError(f"Resume configuration changed: {key}")\n        if args.holdout_already_reviewed != cfg.get("holdout_previously_reviewed", False):\n            raise ValueError("Resume configuration changed: holdout review status")\n    y = tr[TARGET].to_numpy(dtype=int)\n    dev, sealed, cv, outer = split_plan(y, cfg["seed"])\n    cfg = dict(cfg, _cache_dir=str(run / "cache"))\n    return tr, te, sub, id_col, features, run, cfg, y, dev, sealed, cv, outer\n\n\ndef frame_digest(x):\n    h = hashlib.sha256()\n    h.update(repr([(str(c), str(t)) for c, t in x.dtypes.items()]).encode())\n    h.update(pd.util.hash_pandas_object(x, index=False).to_numpy().tobytes())\n    return h.hexdigest()\n\n\ndef prediction_batch(x, y, xp, candidate, cfg, seeds, valid_y=None, model_dir=None):\n    """Checkpoint each exact fit. Trial numbers and selection metadata are not model inputs."""\n    family, variant = candidate["family"], candidate["variant"]\n    fixed = candidate.get("fixed_rounds") if valid_y is None else None\n    cache_dir = Path(cfg["_cache_dir"]) if cfg.get("_cache_dir") else None\n    signature = None\n    if cache_dir:\n        cache_dir.mkdir(parents=True, exist_ok=True)\n        signature = dict(x=frame_digest(x), xp=frame_digest(xp),\n            y=hashlib.sha256(np.asarray(y, dtype=np.int64).tobytes()).hexdigest(),\n            valid_y=None if valid_y is None else hashlib.sha256(np.asarray(valid_y, dtype=np.int64).tobytes()).hexdigest(),\n            family=family, variant=variant, params=candidate["params"], fixed_rounds=fixed,\n            config={k: v for k, v in cfg.items() if not k.startswith("_") and k not in ("seeds", "families", "profile")},\n            implementation=digest(__file__))\n    preds, rounds = [], []\n    prepared = None\n    seed_dependent = variant in ("target", "multiscale_domain") or variant in SIGNAL_VARIANTS\n    for seed in seeds:\n        path = meta_path = None\n        if signature is not None:\n            key = hashlib.sha256(json.dumps(dict(signature, model_seed=seed), sort_keys=True).encode()).hexdigest()\n            path, meta_path = cache_dir / f"{key}.npz", cache_dir / f"{key}.json"\n        model_path = Path(model_dir) / f"seed_{seed}.joblib" if model_dir is not None else None\n        if path is not None and path.exists() and meta_path.exists() and (model_path is None or model_path.exists()):\n            meta = read_json(meta_path)\n            if meta["sha256"] != digest(path):\n                raise ValueError(f"Corrupt prediction cache: {path}")\n            with np.load(path, allow_pickle=False) as saved:\n                values = saved["prediction"]\n            if len(values) != len(xp) or not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():\n                raise ValueError(f"Invalid prediction cache: {path}")\n            preds.append(values)\n            rounds.append(int(meta["rounds"]))\n            print(f"  Reused {family}/{variant} seed={seed}", flush=True)\n            continue\n        if prepared is None or seed_dependent:\n            fe = make_features(variant, seed)\n            xt, xv = fe.fit_transform(x, y), fe.transform(xp)\n            prepared = (fe, xt, xv)\n        else:\n            fe, xt, xv = prepared\n        model, n = fit_model(family, candidate["params"], xt, y,\n            None if valid_y is None else (xv, valid_y), seed, cfg, fixed)\n        values = predict(model, xv, family)\n        if model_path is not None:\n            import joblib\n            model_path.parent.mkdir(parents=True, exist_ok=True)\n            joblib.dump({"features": fe, "model": model, "family": family}, model_path)\n        if path is not None:\n            temporary = path.with_suffix(".tmp")\n            with temporary.open("wb") as stream:\n                np.savez_compressed(stream, prediction=values)\n            temporary.replace(path)\n            write_json(meta_path, {"rounds": n, "sha256": digest(path)})\n        preds.append(values)\n        rounds.append(n)\n    return np.mean(preds, axis=0), rounds\n\n\ndef run_cv(x, y, cv, candidate, cfg, seeds):\n    oof = np.full(len(y), np.nan)\n    rounds, scores = [], []\n    family, variant, params = [candidate[k] for k in ("family", "variant", "params")]\n    for fold, (it, iv) in enumerate(cv):\n        oof[iv], fold_rounds = prediction_batch(x.iloc[it], y[it], x.iloc[iv], candidate, cfg, seeds, valid_y=y[iv])\n        rounds.extend(fold_rounds)\n        scores.append(float(roc_auc_score(y[iv], oof[iv])))\n        print(f"  {family}/{variant} fold={fold} AUC={scores[-1]:.6f}", flush=True)\n    return oof, rounds, scores\n\n\ndef search(args):\n    tr, te, sub, id_col, cols, run, cfg, y, dev, sealed, cv, outer = context(args, True)\n    if (run / "frozen.json").exists() or (run / "SEALED_OPENED.json").exists():\n        raise RuntimeError("This experiment is frozen; search is disabled.")\n    import optuna\n    for family in cfg["families"]:\n        study = optuna.create_study(study_name=family, storage=f"sqlite:///{(run / \'optuna.db\').resolve().as_posix()}",\n            load_if_exists=True, direction="maximize", sampler=optuna.samplers.TPESampler(seed=cfg["seed"]))\n        # Test encodings at the same baseline parameters before the wider search.\n        if not study.trials:\n            variants = ("raw", *SIGNAL_VARIANTS, "multiscale_dual") if cfg["profile"] == "signals" else ("raw", "frequency", "target", "interaction")\n            for index, variant in enumerate(variants):\n                params = defaults(family)\n                if cfg["profile"] == "signals" and index == 5 and family == "lgb":\n                    params.update(max_depth=5, num_leaves=31, min_child_samples=20,\n                        colsample_bytree=0.5, reg_alpha=0.07, reg_lambda=2., max_bin=511)\n                study.enqueue_trial({"variant": variant, **{k: v for k, v in params.items() if k not in ("subsample_freq", "bootstrap_type")}})\n        study.sampler = optuna.samplers.TPESampler(seed=cfg["seed"] + len(study.trials))\n        def objective(trial):\n            started = time.time()\n            variant = trial.suggest_categorical("variant", ["raw", "frequency", "interaction", "artifact", "target", *SIGNAL_VARIANTS])\n            candidate = dict(family=family, variant=variant, params=suggest(trial, family), trial=trial.number)\n            print(f"Starting {family} trial={trial.number}, features={variant}; search seed={cfg[\'seed\']}", flush=True)\n            oof, rounds, scores = run_cv(tr[cols], y, cv, candidate, cfg, [cfg["seed"]])\n            name = f"{family}_{trial.number}"\n            np.save(run / f"{name}_oof.npy", oof)\n            candidate.update(rounds=rounds, fold_auc=scores, dev_auc=float(roc_auc_score(y[dev], oof[dev])), seconds=time.time() - started,\n                actual_device=lgb_effective_device(candidate[\'params\'], cfg) if family == \'lgb\' else (cfg.get(\'xgb_device\', \'cpu\') if family == \'xgb\' else \'cpu\'))\n            write_json(run / f"{name}.json", candidate)\n            trial.set_user_attr("artifact", name)\n            return candidate["dev_auc"]\n        # trials is a TOTAL completed-trial budget per family, making resume idempotent.\n        completed = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])\n        study.optimize(objective, n_trials=max(0, args.trials - completed))\n    print("Search complete. Sealed scores have NOT been computed.", flush=True)\n\n\ndef blend_weights(y, matrix):\n    if matrix.shape[1] == 1:\n        return np.ones(1), float(roc_auc_score(y, matrix[:, 0]))\n    # Small greedy convex search: 20 steps, no unconstrained optimizer.\n    total = np.zeros(len(y))\n    counts = np.zeros(matrix.shape[1])\n    best_score, best = -1, None\n    for step in range(20):\n        scores = [roc_auc_score(y, (total + matrix[:, j]) / (step + 1)) for j in range(matrix.shape[1])]\n        j = int(np.argmax(scores))\n        counts[j] += 1\n        total += matrix[:, j]\n        if scores[j] > best_score:\n            best_score, best = float(scores[j]), counts.copy() / (step + 1)\n    return best, best_score\n\n\ndef freeze(args):\n    tr, te, sub, id_col, cols, run, cfg, y, dev, sealed, cv, outer = context(args)\n    if (run / "frozen.json").exists():\n        print("Already frozen; no changes made.")\n        return\n    if (run / "SEALED_OPENED.json").exists():\n        raise RuntimeError("Sealed fold already opened")\n    candidates = []\n    for family in cfg["families"]:\n        records = [read_json(p) for p in run.glob(f"{family}_[0-9]*.json")]\n        if not records:\n            raise RuntimeError(f"Run search for {family} first")\n        candidates.append(max(records, key=lambda c: c["dev_auc"]))\n    # A fixed, reproducible comparator, independent of the search winners.\n    baseline = dict(family="lgb", variant="raw", params=defaults("lgb"), trial=-1)\n    all_candidates = candidates + [baseline]\n    matrix = []\n    for i, c in enumerate(all_candidates):\n        cached = run / f"{c[\'family\']}_{c[\'trial\']}_oof.npy"\n        if c["trial"] == -1:\n            record_path = run / "lgb_0.json"\n            if record_path.exists():\n                record = read_json(record_path)\n                if record["variant"] == "raw" and record["params"] == c["params"]:\n                    cached = run / "lgb_0_oof.npy"\n                    c["rounds"] = record["rounds"]\n                    c["fold_auc"] = record["fold_auc"]\n        if cfg["seeds"] == [cfg["seed"]] and cached.exists():\n            oof, rounds, scores = np.load(cached), c["rounds"], c["fold_auc"]\n        else:\n            oof, rounds, scores = run_cv(tr[cols], y, cv, c, cfg, cfg["seeds"])\n        c["fixed_rounds"] = int(np.median(rounds))\n        c["seed_dev_auc"] = float(roc_auc_score(y[dev], oof[dev]))\n        c["seed_fold_auc"] = scores\n        matrix.append(oof[dev])\n        np.save(run / f"frozen_candidate_{i}_dev_oof.npy", oof)\n    matrix = np.column_stack(matrix)\n    weights, score = blend_weights(y[dev], matrix[:, :-1])\n    write_json(run / "frozen.json", dict(candidates=candidates, baseline=baseline,\n        weights=weights.tolist(), development_selection_auc=score,\n        warning=("Development AUC is selection-biased. This holdout was reviewed previously; its audit is not a fresh independent evaluation."\n                 if cfg.get("holdout_previously_reviewed", False) else\n                 "Development AUC is selection-biased; sealed audit is the independent check."),\n        config_sha256=digest(run / "config.json")))\n    print(f"Frozen dev selection AUC={score:.6f}; weights={weights}. Next: audit.", flush=True)\n\n\ndef fitted_predictions(x, y, xpred, candidate, cfg, model_dir=None):\n    return prediction_batch(x, y, xpred, candidate, cfg, cfg["seeds"], model_dir=model_dir)[0]\n\n\ndef paired_bootstrap(y, p, baseline, n=300):\n    rng = np.random.default_rng(SEED)\n    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)\n    aucs, deltas = [], []\n    for _ in range(n):\n        ix = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])\n        a = roc_auc_score(y[ix], p[ix])\n        aucs.append(a)\n        deltas.append(a - roc_auc_score(y[ix], baseline[ix]))\n    return dict(auc_95_ci=np.quantile(aucs, [0.025, 0.975]).tolist(),\n                delta_95_ci=np.quantile(deltas, [0.025, 0.975]).tolist(), bootstrap_replicates=n)\n\n\ndef audit(args):\n    tr, te, sub, id_col, cols, run, cfg, y, dev, sealed, cv, outer = context(args)\n    frozen = read_json(run / "frozen.json")\n    if frozen["config_sha256"] != digest(run / "config.json"):\n        raise ValueError("Frozen configuration mismatch")\n    mark = run / "SEALED_OPENED.json"\n    if mark.exists() and read_json(mark)["frozen_sha256"] != digest(run / "frozen.json"):\n        raise ValueError("Frozen candidates changed after audit started")\n    if (run / "sealed_report.json").exists():\n        print(json.dumps(read_json(run / "sealed_report.json"), indent=2))\n        return\n    write_json(mark, {"frozen_sha256": digest(run / "frozen.json"), "opened_at": time.time()})\n    # No sealed labels are supplied to fitting, encoding, early stopping or weight selection.\n    both = pd.concat([tr[cols].iloc[sealed], te[cols]], ignore_index=True)\n    preds = [fitted_predictions(tr[cols].iloc[dev], y[dev], both, c, cfg)[:len(sealed)] for c in frozen["candidates"]]\n    p = np.column_stack(preds) @ np.asarray(frozen["weights"])\n    baseline = fitted_predictions(tr[cols].iloc[dev], y[dev], both, frozen["baseline"], cfg)[:len(sealed)]\n    report = dict(sealed_auc=float(roc_auc_score(y[sealed], p)),\n        baseline_sealed_auc=float(roc_auc_score(y[sealed], baseline)), public_lb=None,\n        sealed_rows=len(sealed),\n        holdout_previously_reviewed=cfg.get("holdout_previously_reviewed", False),\n        note=("Public LB is unmeasured. This holdout split was reviewed previously; this is not a fresh independent evaluation. Do not tune using this result."\n              if cfg.get("holdout_previously_reviewed", False) else\n              "Public LB is unmeasured. Do not tune using this sealed result."))\n    report["delta_vs_baseline"] = report["sealed_auc"] - report["baseline_sealed_auc"]\n    report.update(paired_bootstrap(y[sealed], p, baseline))\n    write_json(run / "sealed_report.json", report)\n    print(json.dumps(report, indent=2), flush=True)\n\n\ndef finalize(args):\n    tr, te, sub, id_col, cols, run, cfg, y, dev, sealed, cv, outer = context(args)\n    frozen = read_json(run / "frozen.json")\n    if not (run / "sealed_report.json").exists():\n        raise RuntimeError("Run the frozen sealed audit first")\n    if read_json(run / "SEALED_OPENED.json")["frozen_sha256"] != digest(run / "frozen.json"):\n        raise ValueError("Candidates changed after audit")\n    all_oof, all_test = [], []\n    for ci, (c, weight) in enumerate(zip(frozen["candidates"], frozen["weights"])):\n        if weight == 0:\n            continue\n        oof, test_preds = np.empty(len(y)), []\n        for fold in range(5):\n            it, iv = np.flatnonzero(outer != fold), np.flatnonzero(outer == fold)\n            # Fixed rounds from development CV: no final-fold early stopping or retuning.\n            both = pd.concat([tr[cols].iloc[iv], te[cols]], ignore_index=True)\n            p = fitted_predictions(tr[cols].iloc[it], y[it], both, c, cfg,\n                run / "models" / f"candidate_{ci}_fold_{fold}" if args.save_models else None)\n            oof[iv] = p[:len(iv)]\n            test_preds.append(p[len(iv):])\n            print(f"Final candidate={ci}, fold={fold} done", flush=True)\n        test_pred = np.mean(test_preds, axis=0)\n        all_oof.append(oof * weight)\n        all_test.append(test_pred * weight)\n        np.savez_compressed(run / f"final_candidate_{ci}.npz", oof=oof, test=test_pred)\n    poof, ptest = np.sum(all_oof, axis=0), np.sum(all_test, axis=0)\n    # Align to sample_submission by ID, never assume its row order equals test order.\n    mapped = pd.Series(ptest, index=te[id_col])\n    sub[TARGET] = sub[id_col].map(mapped)\n    if sub[TARGET].isna().any() or not sub[TARGET].between(0, 1).all():\n        raise ValueError("Invalid submission")\n    sub.to_csv(run / "submission.csv", index=False)\n    pd.DataFrame({id_col: tr[id_col], TARGET: y, "fold": outer, "prediction": poof}).to_csv(run / "final_oof.csv", index=False)\n    write_json(run / "final_report.json", dict(final_oof_auc=float(roc_auc_score(y, poof)), public_lb=None,\n        note="Final OOF reuses development-selected configurations and is NOT an independent score estimate.",\n        submission_rows=len(sub), submission_sha256=digest(run / "submission.csv")))\n    print(f"Created {run / \'submission.csv\'}; Public LB remains unmeasured.", flush=True)\n\n\ndef diagnose(args):\n    tr, te, sub, id_col, cols = load_data(args.data)\n    y = tr[TARGET].to_numpy(dtype=int)\n    dev, _, _, _ = split_plan(y)\n    a = tr.iloc[dev][cols].sample(min(30000, len(dev)), random_state=SEED)\n    b = te[cols].sample(min(30000, len(te)), random_state=SEED)\n    x = pd.concat([a, b], ignore_index=True)\n    labels = np.r_[np.zeros(len(a), int), np.ones(len(b), int)]\n    it, iv = train_test_split(np.arange(len(x)), test_size=0.3, stratify=labels, random_state=SEED)\n    fe = Features("raw")\n    xt = fe.fit_transform(x.iloc[it], labels[it])\n    xv = fe.transform(x.iloc[iv])\n    cfg = dict(max_rounds=300, early_stopping=40, threads=args.threads, xgb_device="cpu")\n    # Fixed complexity to avoid optimizing/reporting the same AV holdout.\n    model, _ = fit_model("lgb", defaults("lgb"), xt, labels[it], None, SEED, cfg, 300)\n    p = predict(model, xv, "lgb")\n    report = dict(adversarial_auc=float(roc_auc_score(labels[iv], p)), id_excluded=True,\n        sealed_excluded=True, train_rows=len(tr), test_rows=len(te), features=cols,\n        development_positive_rate=float(y[dev].mean()),\n        development_duplicate_feature_rows=int(tr.iloc[dev][cols].duplicated().sum()),\n        test_duplicate_feature_rows=int(te[cols].duplicated().sum()),\n        note="AV near 0.5 means this classifier found little shift; it does not prove identical distributions.")\n    Path(args.run).mkdir(parents=True, exist_ok=True)\n    write_json(Path(args.run) / "diagnostics.json", report)\n    print(json.dumps(report, indent=2), flush=True)\n\n\ndef recover_run(args):\n    """Copy validated completed v4.1 CUDA trials; requeue interrupted work in a new run."""\n    import shutil\n    import sqlite3\n    import optuna\n    if not args.source_run:\n        raise ValueError("recover-run requires --source-run")\n    source, destination = Path(args.source_run).resolve(), Path(args.run).resolve()\n    if source == destination or source in destination.parents or destination in source.parents:\n        raise ValueError("Recovery needs a separate sibling run directory")\n    old = read_json(source / \'config.json\')\n    if destination.exists():\n        manifest = destination / \'recovery.json\'\n        if manifest.exists() and read_json(manifest)[\'source_config_sha256\'] == digest(source / \'config.json\') and read_json(destination / \'config.json\')[\'code_sha256\'] == digest(__file__):\n            print(\'Recovery already complete; existing destination retained.\', flush=True)\n            return\n        raise ValueError("Recovery destination already exists; it will not be overwritten")\n    if old[\'code_sha256\'] not in RECOVERABLE_CUDA_CODE_HASHES:\n        raise ValueError("Only the known v4.1 source can be recovered without retraining")\n    if old.get(\'lgb_device\') != \'cuda\' or old[\'families\'] != [\'lgb\'] or old.get(\'profile\') != \'signals\' or old[\'seeds\'] != [old[\'seed\']]:\n        raise ValueError("Recovery supports the single-seed LightGBM signals CUDA run")\n    if any((source / name).exists() for name in (\'frozen.json\', \'SEALED_OPENED.json\', \'sealed_report.json\')):\n        raise ValueError("Cannot recover search after candidate freeze or holdout review")\n    hashes = {name: digest(Path(args.data) / name) for name in old[\'hashes\']}\n    if hashes != old[\'hashes\'] or any(importlib.metadata.version(name) != version for name, version in old[\'versions\'].items()):\n        raise ValueError("Recovery data/library versions differ from the completed trials")\n    tr, _, _, _, _ = load_data(args.data)\n    y = tr[TARGET].to_numpy(dtype=int)\n    dev, sealed, cv, _ = split_plan(y, old[\'seed\'])\n    records = []\n    for path in sorted(source.glob(\'lgb_[0-9]*.json\')):\n        record = read_json(path)\n        if record[\'family\'] != \'lgb\' or record[\'params\'].get(\'max_bin\', 255) > LGB_CUDA_BIN_LIMIT:\n            raise ValueError("Cannot reuse completed trials whose effective backend changes")\n        pred_path = source / f"lgb_{record[\'trial\']}_oof.npy"\n        oof = np.load(pred_path, allow_pickle=False)\n        if oof.shape != y.shape or not np.isnan(oof[sealed]).all() or not np.isfinite(oof[dev]).all() or ((oof[dev] < 0) | (oof[dev] > 1)).any():\n            raise ValueError("Invalid recovered OOF or holdout contamination")\n        if len(record[\'rounds\']) != len(cv) or not np.isclose(roc_auc_score(y[dev], oof[dev]), record[\'dev_auc\'], atol=1e-12, rtol=0):\n            raise ValueError("Recovered OOF does not match its score/rounds")\n        records.append((path, pred_path, record))\n    if not records:\n        raise ValueError("No completed trial artifacts to recover")\n    destination.mkdir(parents=True)\n    # SQLite backup is consistent and leaves the original experiment unchanged.\n    with sqlite3.connect((source / \'optuna.db\').as_uri() + \'?mode=ro\', uri=True) as src_db:\n        with sqlite3.connect(destination / \'optuna.db\') as dst_db:\n            src_db.backup(dst_db)\n    study = optuna.load_study(study_name=\'lgb\', storage=f"sqlite:///{(destination / \'optuna.db\').as_posix()}")\n    complete = {t.number: t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE}\n    if set(complete) != {r[\'trial\'] for _, _, r in records}:\n        raise ValueError("Trial database and completed artifacts disagree")\n    for path, pred_path, record in records:\n        trial = complete[record[\'trial\']]\n        if not np.isclose(trial.value, record[\'dev_auc\'], atol=1e-12, rtol=0) or trial.params.get(\'variant\') != record[\'variant\'] or any(record[\'params\'].get(k) != v for k, v in trial.params.items() if k != \'variant\'):\n            raise ValueError("Trial parameters do not match saved predictions")\n        shutil.copyfile(path, destination / path.name)\n        shutil.copyfile(pred_path, destination / pred_path.name)\n    retried = []\n    for trial in study.trials:\n        if trial.state == optuna.trial.TrialState.RUNNING:\n            fixed = trial.system_attrs.get(\'fixed_params\', {})\n            retry_params = dict(fixed, **trial.params)\n            if not retry_params:\n                raise ValueError("Interrupted trial has no parameters to retry")\n            study.tell(trial.number, state=optuna.trial.TrialState.FAIL)\n            study.enqueue_trial(retry_params)\n            retried.append(trial.number)\n    new = dict(old, code_sha256=digest(__file__), lgb_cuda_bin_limit=LGB_CUDA_BIN_LIMIT)\n    if (source / \'diagnostics.json\').is_file():\n        shutil.copyfile(source / \'diagnostics.json\', destination / \'diagnostics.json\')\n    write_json(destination / \'config.json\', new)\n    write_json(destination / \'recovery.json\', dict(source_config_sha256=digest(source / \'config.json\'),\n        copied_trials=sorted(complete), retried_trials=retried, source_run=str(source),\n        note=\'Only completed <=255-bin CUDA trials were retained; >255-bin fits now use CPU.\'))\n    print(f"Recovered {len(complete)} completed trials without retraining; requeued {retried}. Source retained.", flush=True)\n\n\ndef check_device(args):\n    """Small real fit before a long run; never silently substitute CPU for CUDA."""\n    print(f"LightGBM {importlib.metadata.version(\'lightgbm\')}; Python {platform.python_version()}; "\n          f"device={args.lgb_device}; gpu_id={args.lgb_gpu_id}", flush=True)\n    rng = np.random.default_rng(SEED)\n    x = pd.DataFrame(rng.normal(size=(4096, 5)), columns=[f"f{i}" for i in range(5)])\n    x["category"] = pd.Categorical(rng.integers(0, 4, len(x)))\n    y = (x.f0.to_numpy() + x.f1.to_numpy() + rng.normal(size=len(x)) > 0).astype(int)\n    params = defaults("lgb")\n    params.update(max_bin=LGB_CUDA_BIN_LIMIT, min_child_samples=20)\n    cfg = dict(max_rounds=4, early_stopping=2, threads=args.threads,\n               lgb_device=args.lgb_device, lgb_gpu_id=args.lgb_gpu_id)\n    print("CHECK_STAGE: fit begin (max_bin=255, categories, bagging, validation)", flush=True)\n    model, _ = fit_model("lgb", params, x.iloc[:3072], y[:3072], (x.iloc[3072:], y[3072:]), SEED, cfg)\n    print("CHECK_STAGE: fit completed; predict begin", flush=True)\n    predict(model, x.iloc[3072:], "lgb")\n    print("CHECK_STAGE: predict completed", flush=True)\n    actual = model.booster_.params.get("device_type", "cpu")\n    if actual != args.lgb_device:\n        raise RuntimeError(f"Requested {args.lgb_device}, but got {actual}")\n    print(f"LightGBM backend check passed: {actual}, max_bin=255; larger-bin candidates use CPU", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("command", choices=["recover-run", "check-device", "diagnose", "search", "freeze", "audit", "finalize"])\n    parser.add_argument("--source-run", help="Stopped v4.1 run to copy with recover-run")\n    parser.add_argument("--data", default="/kaggle/input/competitions/playground-series-s6e9")\n    parser.add_argument("--run", default="runs/signals_v4")\n    parser.add_argument("--profile", choices=["signals", "legacy"], default="signals")\n    parser.add_argument("--trials", type=int, default=6, help="Total completed trials per family; increase to resume")\n    parser.add_argument("--families", nargs="+", choices=["xgb", "cat", "lgb"], default=["lgb"])\n    parser.add_argument("--max-rounds", type=int, default=3500)\n    parser.add_argument("--early-stopping", type=int, default=120)\n    parser.add_argument("--threads", type=int, default=min(4, os.cpu_count() or 1))\n    parser.add_argument("--xgb-device", choices=["cpu", "cuda"], default="cpu")\n    parser.add_argument("--lgb-device", choices=["cpu", "cuda"], default="cpu")\n    parser.add_argument("--lgb-gpu-id", type=int, default=0, help="Single CUDA device used by LightGBM")\n    parser.add_argument("--seeds", nargs="+", type=int, default=[2026],\n                        help="Seeds used after model selection; search always uses the single split seed")\n    parser.add_argument("--holdout-already-reviewed", action="store_true",\n                        help="Mark a reused holdout as previously reviewed, not fresh independent evidence")\n    parser.add_argument("--save-models", action="store_true")\n    args = parser.parse_args()\n    if min(args.trials, args.max_rounds, args.early_stopping, args.threads) < 1:\n        parser.error("Numeric budgets must be positive")\n    if args.lgb_gpu_id < 0:\n        parser.error("GPU device ID must be nonnegative")\n    if len(set(args.seeds)) != len(args.seeds) or len(set(args.families)) != len(args.families):\n        parser.error("Duplicate seeds/families")\n    globals()[args.command.replace("-", "_")](args)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
GPU_SETUP = Path('/kaggle/working/gpu_setup.py')
GPU_SETUP.write_text('"""Kaggle/Linux CUDA setup. All probes run in child processes to avoid stale DLLs.\n\nBuild flags follow https://github.com/lightgbm-org/LightGBM/blob/main/python-package/README.rst\n"""\nimport importlib.metadata\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport shutil\nimport subprocess\nimport sys\n\nCUDA_LIGHTGBM_VERSION = "4.7.0"\nWHEEL_FOLDER = "lightgbm_cuda_wheels"\n\n\ndef detect_gpu_ids(max_gpus=2):\n    """Return physical selectors for child CUDA_VISIBLE_DEVICES, preserving restrictions."""\n    if max_gpus < 1:\n        raise ValueError(\'max_gpus must be positive\')\n    visible = os.environ.get(\'CUDA_VISIBLE_DEVICES\')\n    if visible is not None:\n        selectors = []\n        for token in visible.split(\',\'):\n            token = token.strip()\n            if token.isdigit() or token.startswith((\'GPU-\', \'MIG-\')):\n                if token in selectors:\n                    break\n                selectors.append(token)\n            else:\n                # CUDA stops enumeration at the first invalid token, including -1.\n                break\n    else:\n        try:\n            result = subprocess.run([\'nvidia-smi\', \'--query-gpu=uuid\', \'--format=csv,noheader\'],\n                check=True, text=True, capture_output=True)\n            selectors = [s.strip() for s in result.stdout.splitlines() if s.strip().startswith(\'GPU-\')]\n        except (FileNotFoundError, subprocess.CalledProcessError):\n            selectors = []\n    if not selectors:\n        raise RuntimeError(\'No visible NVIDIA GPU. Enable Kaggle Settings > Accelerator and check CUDA_VISIBLE_DEVICES.\')\n    if any(s.startswith(\'MIG-\') for s in selectors):\n        return selectors[:1]\n    return selectors[:max_gpus]\n\n\ndef require_t4_pair(selectors=None):\n    """Require two visible T4 devices and return their physical CUDA selectors."""\n    selectors = detect_gpu_ids(2) if selectors is None else list(map(str, selectors))\n    if len(selectors) != 2 or len(set(selectors)) != 2:\n        raise RuntimeError(\'This Notebook requires exactly two visible T4 GPUs. Select GPU T4 x2 in Kaggle.\')\n    try:\n        result = subprocess.run([\'nvidia-smi\', \'--query-gpu=index,uuid,name\', \'--format=csv,noheader\'],\n            check=True, text=True, capture_output=True)\n    except (FileNotFoundError, subprocess.CalledProcessError) as exc:\n        raise RuntimeError(\'Cannot inspect the required T4 x2 accelerator with nvidia-smi.\') from exc\n    inventory = [tuple(part.strip() for part in line.split(\',\', 2))\n                 for line in result.stdout.splitlines() if line.strip()]\n    names = []\n    for selector in selectors:\n        matches = [name for index, uuid, name in inventory\n                   if selector == index or uuid.startswith(selector) or selector.startswith(uuid)]\n        if len(matches) != 1:\n            raise RuntimeError(f\'Cannot resolve visible GPU selector {selector!r}.\')\n        names.append(matches[0])\n    if any(\'T4\' not in name.upper() for name in names):\n        raise RuntimeError(f\'This Notebook requires T4 x2; detected {names}.\')\n    return selectors\n\n\ndef _probe(command, env, log_path):\n    result = subprocess.run(command, text=True, capture_output=True, env=env)\n    output = (result.stdout or "") + (result.stderr or "")\n    Path(log_path).write_text(f"returncode={result.returncode}\\n{output}", encoding="utf-8")\n    if output:\n        print(output, end="", flush=True)\n    return result, output\n\n\ndef _failed(result, log_path):\n    signal = f"native crash (signal {-result.returncode})" if result.returncode < 0 else f"exit {result.returncode}"\n    raise RuntimeError(f"LightGBM backend test failed: {signal}; no CPU fallback was applied. "\n        f"Full diagnostic log: {log_path}. Share this log before trying another rebuild.")\n\n\ndef _digest(path):\n    value = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            value.update(chunk)\n    return value.hexdigest()\n\n\ndef _wheel_marker(wheel):\n    return dict(version=CUDA_LIGHTGBM_VERSION, architecture=\'native\',\n        machine=platform.machine(), python=[sys.version_info.major, sys.version_info.minor],\n        wheel=Path(wheel).name, sha256=_digest(wheel))\n\n\ndef _cached_cuda_wheels(folder):\n    markers = [folder / WHEEL_FOLDER / \'lightgbm_cuda_wheel.json\']\n    kaggle_inputs = Path(\'/kaggle/input\')\n    if kaggle_inputs.exists():\n        markers.extend(kaggle_inputs.glob(f\'*/{WHEEL_FOLDER}/lightgbm_cuda_wheel.json\'))\n    expected = dict(version=CUDA_LIGHTGBM_VERSION, architecture=\'native\',\n        machine=platform.machine(), python=[sys.version_info.major, sys.version_info.minor])\n    for marker_path in markers:\n        if not marker_path.exists():\n            continue\n        try:\n            marker = json.loads(marker_path.read_text(encoding=\'utf-8\'))\n            wheel = marker_path.parent / marker[\'wheel\']\n        except (KeyError, ValueError, OSError):\n            continue\n        if all(marker.get(name) == value for name, value in expected.items()) \\\n                and wheel.is_file() and marker.get(\'sha256\') == _digest(wheel):\n            yield wheel\n\n\ndef ensure_lightgbm_backend(script, device="cuda", gpu_id=0, build_if_missing=True, visible_devices=None):\n    if device not in ("cpu", "cuda"):\n        raise ValueError("Choose cpu or cuda")\n    command = [sys.executable, "-X", "faulthandler", "-u", str(script), "check-device", "--lgb-device", device,\n               "--lgb-gpu-id", str(gpu_id), "--threads", "2"]\n    folder = Path(script).resolve().parent\n    env = os.environ.copy()\n    if visible_devices is not None:\n        env[\'CUDA_VISIBLE_DEVICES\'] = str(visible_devices)\n    env["PYTHONFAULTHANDLER"] = "1"\n    env["PYTHONUNBUFFERED"] = "1"\n    if device == "cuda":\n        if platform.system() != "Linux":\n            raise RuntimeError("Use LightGBM CUDA in the Kaggle Linux GPU session. On this computer use LGB_DEVICE=\'cpu\'.")\n        smi = shutil.which("nvidia-smi")\n        if not smi:\n            raise RuntimeError("Enable a GPU in Kaggle Settings > Accelerator, then rerun this cell.")\n        subprocess.run([smi, "--query-gpu=index,name", "--format=csv,noheader"], check=True, env=env)\n    log_path = folder / "gpu_preflight_before.log"\n    probe, failure = _probe(command, env, log_path)\n    if probe.returncode == 0:\n        return\n    # One bounded repair for absent CUDA support or a native crash; keep other errors visible.\n    not_compiled = "CUDA Tree Learner was not enabled in this build" in failure\n    native_crash = probe.returncode in (-11, -6)\n    if device != "cuda" or not build_if_missing or not (not_compiled or native_crash):\n        _failed(probe, log_path)\n    for wheel in _cached_cuda_wheels(folder):\n        print(f\'Reusing cached CUDA LightGBM wheel: {wheel}\', flush=True)\n        subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'--force-reinstall\', \'--no-deps\', str(wheel)],\n            check=True, env=env)\n        cached_log = folder / \'gpu_preflight_cached.log\'\n        probe, failure = _probe(command, env, cached_log)\n        log_path = cached_log\n        if probe.returncode == 0:\n            return\n        not_compiled = "CUDA Tree Learner was not enabled in this build" in failure\n        native_crash = probe.returncode in (-11, -6)\n        if not (not_compiled or native_crash):\n            _failed(probe, cached_log)\n    attempt = folder / "gpu_cuda_repair.json"\n    repair = dict(version=CUDA_LIGHTGBM_VERSION, architecture="native", gpu_id=gpu_id)\n    if attempt.exists() and json.loads(attempt.read_text(encoding="utf-8")) == repair:\n        _failed(probe, log_path)\n    cuda_bin = Path(os.environ.get("CUDA_HOME", "/usr/local/cuda")) / "bin"\n    env["PATH"] = str(cuda_bin) + os.pathsep + env.get("PATH", "")\n    if not shutil.which("nvcc", path=env["PATH"]):\n        raise RuntimeError("CUDA compiler nvcc is unavailable. Use a Kaggle GPU runtime with CUDA Toolkit.")\n    env.setdefault("CMAKE_BUILD_PARALLEL_LEVEL", "2")\n    version = CUDA_LIGHTGBM_VERSION\n    wheel_folder = folder / WHEEL_FOLDER\n    wheel_folder.mkdir(parents=True, exist_ok=True)\n    build = [sys.executable, "-m", "pip", "wheel", "--no-deps",\n        "--no-cache-dir", "--no-binary=lightgbm", "--config-settings=cmake.define.USE_CUDA=ON",\n        "--config-settings=cmake.define.CMAKE_CUDA_ARCHITECTURES=native",\n        "--wheel-dir", str(wheel_folder), f"lightgbm=={version}"]\n    print(f"Building LightGBM {version} with CUDA. Enable Kaggle Internet; this first setup can take several minutes.", flush=True)\n    subprocess.run(build, check=True, env=env)\n    wheels = sorted(wheel_folder.glob(f\'lightgbm-{version}-*.whl\'), key=lambda path: path.stat().st_mtime)\n    if not wheels:\n        raise RuntimeError(f\'CUDA LightGBM wheel was not created in {wheel_folder}.\')\n    wheel = wheels[-1]\n    subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'--force-reinstall\', \'--no-deps\', str(wheel)],\n        check=True, env=env)\n    (wheel_folder / \'lightgbm_cuda_wheel.json\').write_text(\n        json.dumps(_wheel_marker(wheel), sort_keys=True), encoding=\'utf-8\')\n    attempt.write_text(json.dumps(repair), encoding="utf-8")\n    # A fresh interpreter loads the newly installed native library.\n    log_path = folder / "gpu_preflight_after.log"\n    probe, _ = _probe(command, env, log_path)\n    if probe.returncode:\n        _failed(probe, log_path)\n', encoding='utf-8')
DIVERSITY_SCRIPT = Path('/kaggle/working/diversity.py')
DIVERSITY_SCRIPT.write_text('"""Compatibility CLI for the package-based T4 x2 pipeline."""\nimport argparse\nimport subprocess\n\nimport joblib\nimport numpy as np\nimport optuna\nfrom sklearn.metrics import roc_auc_score\n\nimport prototype as p\nfrom gpu_setup import require_t4_pair\nfrom s6e9.artifact import (checked_prediction, key, write_runtime_profile,\n                           write_screen_diagnostics)\nfrom s6e9.audit import audit\nfrom s6e9.finalize import finalize\nfrom s6e9.freeze import (blend_values, crossfit_blend, fold_tie_statistics, freeze,\n                         frozen_config, gpu_primary_gate, improvement_gate,\n                         lexicographic_oof, lexicographic_rank, load_candidate,\n                         rank_blend_allowed, rank_columns, rank_oof_matrix,\n                         tie_break_allowed, weight_grid)\nfrom s6e9.tuning import (anchor, cat_anchor, context, domain_anchor, evaluate, folds,\n                         gpu_pool, realmlp_anchor, search, shutdown_gpu_workers,\n                         single_seed_folds, suggest, worker, worker_loop)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'command\', choices=[\'search\', \'freeze\', \'audit\', \'finalize\', \'worker\', \'worker-loop\'])\n    parser.add_argument(\'--job\')\n    parser.add_argument(\'--data\', default=\'/kaggle/input/competitions/playground-series-s6e9\')\n    parser.add_argument(\'--run\', default=\'/kaggle/working/s6e9_diversity_v6_0\')\n    parser.add_argument(\'--seed\', type=int, default=2026)\n    parser.add_argument(\'--threads\', type=int, default=4, help=\'Total CPU budget across concurrent folds\')\n    parser.add_argument(\'--gpu-ids\', nargs=\'+\', default=None, help=\'Two physical T4 IDs/UUIDs; default auto-detect\')\n    parser.add_argument(\'--domain-compare\', action=argparse.BooleanOptionalAction, default=True)\n    parser.add_argument(\'--cat-compare\', action=argparse.BooleanOptionalAction, default=True)\n    parser.add_argument(\'--realmlp-compare\', action=argparse.BooleanOptionalAction, default=True)\n    parser.add_argument(\'--final-seeds\', nargs=\'+\', type=int, default=[2026, 42, 3407],\n                        help=\'Average these seeds only when the selected primary GPU model passes the OOF gate\')\n    parser.add_argument(\'--screen-folds\', type=int, default=2)\n    parser.add_argument(\'--promote-trials\', type=int, default=3)\n    parser.add_argument(\'--max-rounds\', type=int, default=3500)\n    parser.add_argument(\'--early-stopping\', type=int, default=120)\n    parser.add_argument(\'--lgb-trials\', type=int, default=12)\n    parser.add_argument(\'--xgb-trials\', type=int, default=6)\n    parser.add_argument(\'--auc-window\', type=float, default=.0004)\n    parser.add_argument(\'--max-corr\', type=float, default=.999)\n    args = parser.parse_args()\n    if args.command == \'worker\':\n        if not args.job:\n            parser.error(\'worker requires --job\')\n        worker(args.job)\n        return\n    if args.command == \'worker-loop\':\n        worker_loop()\n        return\n    if min(args.threads, args.max_rounds, args.early_stopping) < 1 or min(args.lgb_trials, args.xgb_trials) < 0:\n        parser.error(\'Invalid budgets\')\n    if not 0 < args.max_corr <= 1 or args.auc_window < 0 or args.auc_window > 1:\n        parser.error(\'Invalid candidate filtering thresholds\')\n    if not 1 <= args.screen_folds <= 4 or args.promote_trials < 1:\n        parser.error(\'screen-folds must be 1..4 and promote-trials must be positive\')\n    if not args.final_seeds or len(set(args.final_seeds)) != len(args.final_seeds):\n        parser.error(\'final-seeds must be a non-empty unique list\')\n    args.gpu_ids = require_t4_pair(args.gpu_ids)\n    try:\n        globals()[args.command](args)\n    finally:\n        shutdown_gpu_workers()\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
_spec = importlib.util.spec_from_file_location('s6e9_gpu_setup', GPU_SETUP)
_gpu_setup = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_gpu_setup)
GPU_IDS = _gpu_setup.require_t4_pair(GPU_IDS)
migrate_restored_run(RUN, GPU_IDS)
print('Validated T4 selectors:', GPU_IDS)
for selector in GPU_IDS:
    _gpu_setup.ensure_lightgbm_backend(SCRIPT, 'cuda', 0, BUILD_CUDA_IF_NEEDED, visible_devices=selector)
print('Training package ready:', PACKAGE)


In [ ]:
def execute(command):
    args = [sys.executable, '-u', str(DIVERSITY_SCRIPT), command,
        '--data', str(DATA), '--run', str(RUN), '--seed', str(SEED),
        '--gpu-ids', *map(str, GPU_IDS),
        '--threads', str(THREADS), '--max-rounds', str(MAX_ROUNDS),
        '--early-stopping', str(EARLY_STOPPING),
        '--lgb-trials', str(LGB_TRIALS), '--xgb-trials', str(XGB_TRIALS),
        '--screen-folds', str(SCREEN_FOLDS), '--promote-trials', str(PROMOTE_TRIALS),
        '--final-seeds', *map(str, FINAL_SEEDS),
        '--domain-compare' if DOMAIN_COMPARE else '--no-domain-compare',
        '--cat-compare' if CAT_COMPARE else '--no-cat-compare',
        '--realmlp-compare' if REALMLP_COMPARE else '--no-realmlp-compare']
    subprocess.run(args, check=True)


## 探索
木モデルと固定RealMLP候補をdevelopment foldで評価します。


In [ ]:
execute('search')


## 候補固定
seed平均と広いblendはcross-fold評価で改善した場合だけ採用します。


In [ ]:
execute('freeze')


## Holdout確認
固定済み構成を参考評価します。


In [ ]:
execute('audit')


## 最終学習
5-fold予測と提出CSVを作ります。


In [ ]:
execute('finalize')
import json
import shutil
from pathlib import Path
import pandas as pd
from IPython.display import FileLink, display


def export_submission(run, output_dir='/kaggle/working'):
    source = Path(run) / 'submission.csv'
    if not source.is_file():
        raise FileNotFoundError(f'学習結果がありません: {source}。finalizeの完了を確認してください。')
    destination = Path(output_dir) / 'submission.csv'
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source.resolve() != destination.resolve():
        shutil.copyfile(source, destination)
    return destination


# Kaggleの提出画面が選択できるよう、作業領域の直下にも保存する。
SUBMISSION = export_submission(RUN)
display(pd.read_csv(SUBMISSION).head())
display(FileLink(str(SUBMISSION)))
print('Submission file:', SUBMISSION)
print(json.loads((RUN / 'sealed_report.json').read_text()))
